Author: Sparshita Dey

Last modified: 8th May 2026

---

# CC1e0pi Selection Analysis
**Pandora-only (legacy) | BNB MC Overlays | No trigger**

1. Setup and style
2. Load data and define cuts
    - Using legacy cut definitions as defined by the reco group for NuMI data
    - Data = BNB MC + Overlays, version v10_06_00_06p03 (the most recent "official production" CAF files)
3. Sanity check: reproduce efficiency curve
    - First glance of the performance of NuMI cuts on a BNB dataset 
4. Cut flow table
    - The impact of sequentially applying cuts on the distribution of data as a function of both the true neutrino energy, $E_{\nu}^{true}$, and the reconstructed neutrino energy, $E_{\nu}^{reco}$.

---

5. Electron KE threshold sweep (Pareto map)
    - Typically applied at 0.2 GeV 
6. Individual cut sweeps with Punzi FOM
7. Cut ordering study
8. 2D efficiency/purity maps
9. Neutrino energy resolution and regression

In [ ]:
import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.colors import Normalize
from scipy.ndimage import gaussian_filter
from scipy.stats import linregress
from scipy.optimize import curve_fit
import warnings
from cmcrameri import cm
import matplotlib.patches as mpatches
from matplotlib.ticker import MultipleLocator

warnings.filterwarnings('ignore')

# -- Style ---------------------------------------------------------------------
plt.rcParams.update({
    'font.family':         'serif',
    'font.serif':          ['Times New Roman','DejaVu Serif'],
    'mathtext.fontset':    'cm',
    'axes.labelsize':      14,
    'axes.titlesize':      13,
    'axes.titlepad':       10,
    'xtick.labelsize':     11,
    'ytick.labelsize':     11,
    'legend.fontsize':     10,
    'legend.framealpha':   0.88,
    'legend.edgecolor':    '#cccccc',
    'legend.fancybox':     True,
    'figure.dpi':          140,
    'figure.facecolor':    'white',
    'axes.facecolor':      '#f9f9f9',
    'axes.edgecolor':      '#444444',
    'axes.linewidth':      1.3,
    'axes.grid':           True,
    'grid.color':          '#dddddd',
    'grid.linewidth':      0.7,
    'xtick.direction':     'in',
    'ytick.direction':     'in',
    'xtick.top':           True,
    'ytick.right':         True,
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
    'xtick.major.size':    5,
    'ytick.major.size':    5,
    'xtick.minor.size':    2.5,
    'ytick.minor.size':    2.5,
    'lines.linewidth':     2,
})

CRIMSON  = '#DC143C'
GOLD     = '#DAA520'
BLUE     = '#3a7ebf'
MANAGUA  = cm.managua
SEL_COLS = ['#111111','#8c1d18','#f26b6c','#f39c12',
            '#3f8f3f','#61d6d6','#4257f5','#c13fd4']

def band_plot(ax, x, y, yerr, color, label='', lw=2, ms=6,
              alpha_band=0.18, fmt='o', step=False):
    """Line + marker + shaded error band. Cleaner than individual caps."""
    if np.ndim(yerr)==1:
        ylo, yhi = y-yerr, y+yerr
    else:
        ylo, yhi = y-yerr[0], y+yerr[1]
    if step:
        ax.fill_between(x, ylo, yhi, color=color, alpha=alpha_band,
                        step='mid', zorder=2)
    else:
        ax.fill_between(x, ylo, yhi, color=color, alpha=alpha_band, zorder=2)
    ax.plot(x, y, fmt+'-' if fmt else '-', color=color, lw=lw,
            ms=ms, label=label, zorder=4,
            markeredgecolor='white', markeredgewidth=0.5)

def vline_base(ax, val, color=CRIMSON, label=None):
    ax.axvline(val, ls='--', lw=1.8, color=color, alpha=0.85,
               label=label or f'Baseline ({val})', zorder=6)

def vline_opt(ax, val, color=GOLD, label=None):
    ax.axvline(val, ls=':', lw=1.8, color=color, alpha=0.9,
               label=label or f'FOM opt. ({val:.2f})', zorder=6)

def savefig(fig, fname):
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    print(f'Saved {fname}')

print('Setup complete.')

## 1. Load data
---
Data first run processe through macro: `/exp/icarus/app/users/sdey2/NuGraphReco/cc1e0pi_v3/src/CC1e0piSelection_MakeAll.C` 

This takes ~ 5 hours on gpvms.

In [ ]:
ROOT_FILE = '/exp/icarus/app/users/sdey2/NuGraphReco/cc1e0pi_v3/output/root/CC1e0piSelection_All.root'   

df = uproot.open(ROOT_FILE)['ntuple/ntuple'].arrays(library='pd')

# replace sentinels with NaN
for col in df.columns:
    df[col] = df[col].replace([-5., -9999.], np.nan)

print(f'Slices: {len(df):,}  |  Branches: {len(df.columns)}')

In [ ]:
## 0. BNB CV: topology breakdown vs reconstructed neutrino energy
from cmcrameri import cm as cmc

_f = uproot.open(ROOT_FILE)

_TOPO_KEYS = [
    ('signal',     r'CC1e$^{\pm}$0$\pi$'),
    ('othernuecc', r'$\nu_e$CC'),
    ('nuenc',      r'$\nu_e$NC'),
    ('numucc_pi0', r'$\nu_\mu$CC$\pi^0$'),
    ('numucc',     r'$\nu_\mu$CC'),
    ('numunc',     r'$\nu_\mu$NC'),
    ('oofvnu',     r'OoFV'),
    ('cosmic',     r'Cosmic'),
]
_colors = [cmc.managua(x) for x in np.linspace(0.05, 0.95, len(_TOPO_KEYS))]

_hists, _edges = {}, None
for _key, _ in _TOPO_KEYS:
    _v, _e = _f[f'selection/reconuenergy_{_key}'].to_numpy()
    _hists[_key] = _v
    if _edges is None: _edges = _e

_total, _ = _f['selection/reconuenergy_total'].to_numpy()
_ctrs  = 0.5 * (_edges[:-1] + _edges[1:])
_width = _edges[1] - _edges[0]

fig, ax = plt.subplots(figsize=(10, 5.5), dpi=400)
_bot = np.zeros(len(_ctrs))
for (_key, _label), _color in zip(_TOPO_KEYS, _colors):
    ax.bar(_ctrs, _hists[_key], width=_width, bottom=_bot,
           color=_color, edgecolor='#333333', linewidth=0.45, label=_label, zorder=3)
    _bot += _hists[_key]

_err = np.sqrt(_total)
ax.bar(_ctrs, 2*_err, width=_width, bottom=_total - _err,
       color='none', edgecolor='#444444', linewidth=0,
       hatch='////', alpha=0.35, zorder=4)

_handles = [mpatches.Patch(facecolor=c, edgecolor='#333333', linewidth=0.5, label=l)
            for (_, l), c in zip(_TOPO_KEYS, _colors)]
_handles += [mpatches.Patch(facecolor='none', edgecolor='#444444',
                             hatch='////', alpha=0.5, label='Stat. unc.')]
ax.legend(handles=_handles, title='BNB CV  (no trigger, Pandora)',
          title_fontsize=9, fontsize=9, loc='upper right',
          framealpha=0.92, edgecolor='#cccccc', fancybox=True)

ax.set_xlabel(r'$E_{\nu}^{\mathrm{reco}}$ [GeV]')
ax.set_ylabel(r'Slices / $8.49 \times 10^{20}$ POT')
ax.set_xlim(0, 3)
ax.set_ylim(0, _bot.max() * 1.35)
ax.xaxis.set_minor_locator(MultipleLocator(0.1))
ax.yaxis.set_minor_locator(MultipleLocator(5))
ax.set_title(r'Selected slices by topology vs $E_{\nu}^{\mathrm{reco}}$ — BNB MC overlays')
fig.tight_layout()
plt.show()

## 2. Baseline cuts
---
The nominal cuts used for NuMI analysis to provide a reference point when comparing BNB.

In [ ]:
THRESHOLD_E        = 0.200
SHOWER_ENERGY_MIN  = 0.200
SHOWER_DEDX_MAX    = 3.500
SHOWER_ANGLE_MAX   = 15.0
SHOWER_CONVGAP_MAX = 4.0
MUON_CHI2_MU_MAX   = 30.0
MUON_CHI2_P_MIN    = 80.0

def signal_mask(df, ke=THRESHOLD_E):
    return (df['is_signal']==1) & (df['true_electron_ke'].fillna(0) > ke)

def c_presel(df):             return (df['pass_not_clear_cosmic']==1) & (df['pass_vertex_fv']==1)
def c_flash(df):              return df['pass_flash_match']==1
def c_shower(df, v=SHOWER_ENERGY_MIN):   return df['shower_energy'].fillna(0) > v
def c_dedx(df, v=SHOWER_DEDX_MAX):      return (df['shower_avail_dedx'].fillna(99)<v) & (df['shower_avail_dedx'].fillna(0)>0)
def c_angle(df, v=SHOWER_ANGLE_MAX):    return df['shower_open_angle'].fillna(99) < v
def c_gap(df, v=SHOWER_CONVGAP_MAX):    return df['shower_conv_gap'].fillna(99) < v
def c_proton(df):             return df['n_protons'].fillna(0) >= 1
def c_0pi(df):                return df['n_other_particles'].fillna(99) == 0
def c_muon(df, v=MUON_CHI2_MU_MAX):
    return ~((df['longest_trk_chi2_muon'].fillna(99)<v) & (df['longest_trk_chi2_proton'].fillna(0)>MUON_CHI2_P_MIN))

def sel_presel(df):   return c_presel(df)
def sel_flash(df):    return sel_presel(df)  & c_flash(df)
def sel_shower(df):   return sel_flash(df)   & c_shower(df)
def sel_dedx(df):     return sel_shower(df)  & c_dedx(df)
def sel_anglegap(df): return sel_dedx(df)    & c_angle(df) & c_gap(df)
def sel_proton(df):   return sel_anglegap(df) & c_proton(df)
def sel_0pi(df):      return sel_proton(df)  & c_0pi(df)
def sel_full(df):     return sel_0pi(df)     & c_muon(df)

CUT_STEPS = [
    ('Presel.',         sel_presel),
    ('FM',              sel_flash),
    ('Electron ID',     sel_shower),
    ('dE/dx',           sel_dedx),
    ('Angle, gap',      sel_anglegap),
    ('Proton ID',       sel_proton),
    ('$0\\pi$',          sel_0pi),
    ('LE-$\\mu$ veto',   sel_full),
]

sig = signal_mask(df)
print(f'True signal: {sig.sum():,}  |  Full selection: {sel_full(df).sum():,}')
print(f'Efficiency: {(sel_full(df)&sig).sum()/sig.sum():.3f}  |  Purity: {(sel_full(df)&sig).sum()/sel_full(df).sum():.3f}')

## 3. Efficiency curve (sanity check vs CAFAna)
---

In [ ]:
E_BINS = np.array([0.0,0.2,0.4,0.6,0.8,1.0,1.2,1.4,1.6,1.8,2.0,2.4,2.8,3.2,4.0])
E_CEN  = 0.5*(E_BINS[:-1]+E_BINS[1:])
E_WID  = np.diff(E_BINS)

fig, ax = plt.subplots(figsize=(10, 5.5), dpi=300)

denom, _ = np.histogram(df.loc[sig,'true_nu_energy'].dropna(), bins=E_BINS)

for (label, cut_fn), col in zip(CUT_STEPS, SEL_COLS):
    mask = cut_fn(df) & sig
    num, _ = np.histogram(df.loc[mask,'true_nu_energy'].dropna(), bins=E_BINS)
    eff  = np.where(denom>0, num/denom, np.nan)
    err  = np.where(denom>0, np.sqrt(eff*(1-eff)/np.where(denom>0,denom,1)), np.nan)
    tot  = mask.sum()/sig.sum()

    # shaded band instead of caps
    ax.fill_between(E_CEN, eff-err, eff+err,
                    color=col, alpha=0.15, step='mid', zorder=2)
    ax.errorbar(E_CEN, eff, xerr=E_WID/2,
                fmt='o', ms=4.5, lw=1.8, capsize=0, color=col,
                markeredgecolor='white', markeredgewidth=0.5,
                label=f'{label} ({100*tot:.0f}%)', zorder=5)

# truth backdrop on twin axis
ax2 = ax.twinx()
ax2.hist(df.loc[sig,'true_nu_energy'].dropna(), bins=E_BINS,
         color='#bdbdbd', alpha=0.22, zorder=0)
ax2.set_ylabel('Signal events', color='#999999', fontsize=11)
ax2.tick_params(axis='y', colors='#999999')
ax2.set_ylim(bottom=0)
ax2.grid(False)

ax.axhline(1.0, ls='--', lw=1.5, color='#555555', zorder=5)
ax.set_xlabel(r'$E_{\nu}$ [GeV]')
ax.set_ylabel('Selection efficiency')
ax.set_xlim(0, 4); ax.set_ylim(0, 1.35)
ax.legend(ncol=3, loc='upper right', fontsize=9)
ax.set_title('CC1e0pi selection efficiency (no trigger, Pandora-only)')
fig.tight_layout()
#savefig(fig, 'efficiency_python.png')
plt.show()

### Selection purity vs $E_{\nu}^{\mathrm{true}}$

Companion to the efficiency curve above.  Purity is defined bin-by-bin as (selected signal) / (all selected events); the cumulative purity at each cut stage is shown to match the style of the NuMI CAFAna plots.

In [ ]:
# Selection purity vs true neutrino energy (companion to the efficiency curve above).
# Purity = (selected signal in bin) / (all selected events in bin), cumulative at each cut stage.

E_BINS_P = np.array([0.0,0.2,0.4,0.6,0.8,1.0,1.2,1.4,1.6,1.8,2.0,2.4,2.8,3.2,4.0])
E_CEN_P  = 0.5*(E_BINS_P[:-1]+E_BINS_P[1:])
E_WID_P  = np.diff(E_BINS_P)

fig, ax = plt.subplots(figsize=(10, 5.5), dpi=300)

for (label, cut_fn), col in zip(CUT_STEPS, SEL_COLS):
    mask   = cut_fn(df)
    n_sig_b, _ = np.histogram(df.loc[mask & sig, 'true_nu_energy'].dropna(), bins=E_BINS_P)
    n_sel_b, _ = np.histogram(df.loc[mask,        'true_nu_energy'].dropna(), bins=E_BINS_P)
    pur  = np.where(n_sel_b > 0, n_sig_b / n_sel_b, np.nan)
    err  = np.where(n_sel_b > 0, np.sqrt(pur*(1-pur)/np.where(n_sel_b>0, n_sel_b, 1)), np.nan)
    tot_pur = (mask & sig).sum() / mask.sum() if mask.sum() > 0 else np.nan

    ax.fill_between(E_CEN_P, pur-err, pur+err,
                    color=col, alpha=0.15, step='mid', zorder=2)
    ax.errorbar(E_CEN_P, pur, xerr=E_WID_P/2,
                fmt='o', ms=4.5, lw=1.8, capsize=0, color=col,
                markeredgecolor='white', markeredgewidth=0.5,
                label=f'{label} ({100*tot_pur:.0f}%)', zorder=5)

# Selected-event backdrop on twin axis (using the full selection)
ax2 = ax.twinx()
ax2.hist(df.loc[sel_full(df), 'true_nu_energy'].dropna(), bins=E_BINS_P,
         color='#bdbdbd', alpha=0.22, zorder=0)
ax2.set_ylabel('Selected events (full sel.)', color='#999999', fontsize=11)
ax2.tick_params(axis='y', colors='#999999')
ax2.set_ylim(bottom=0)
ax2.grid(False)

ax.axhline(1.0, ls='--', lw=1.5, color='#555555', zorder=5)
ax.set_xlabel(r'$E_{\nu}^{\mathrm{true}}$ [GeV]')
ax.set_ylabel('Selection purity')
ax.set_xlim(0, 4); ax.set_ylim(0, 1.35)
ax.legend(ncol=3, loc='upper left', fontsize=9)
ax.set_title('CC1e0pi selection purity (no trigger, Pandora-only)')
fig.tight_layout()
plt.show()

## 4. Cut flow table
---
Summary of the impact of each sequential cut on selection (statistics, efficiencies, purities). 

In [ ]:
rows = []
n_sig_tot = sig.sum()
for label, cut_fn in CUT_STEPS:
    sel   = cut_fn(df)
    n_sel = sel.sum()
    n_sig = (sel & sig).sum()
    n_bkg = n_sel - n_sig
    eff   = n_sig/n_sig_tot if n_sig_tot>0 else 0
    pur   = n_sig/n_sel     if n_sel>0     else 0
    punzi = eff/(1+np.sqrt(max(n_bkg,0)))
    rows.append({'Cut': label.replace('$','').replace('\\',''),
                 'N sel': f'{n_sel:,}', 'N sig': f'{n_sig:,}',
                 'Eff (%)': f'{100*eff:.1f}',
                 'Pur (%)': f'{100*pur:.1f}',
                 'Punzi FOM': f'{punzi:.5f}'})

cf = pd.DataFrame(rows)
print(cf.to_string(index=False))

# -- Visual cut flow bar chart -------------------------------------------------
effs_cf = [float(r['Eff (%)']) for r in rows]
purs_cf = [float(r['Pur (%)']) for r in rows]
labels_cf = [r['Cut'] for r in rows]
x = np.arange(len(labels_cf))

fig, ax = plt.subplots(figsize=(11, 5), dpi=300)
ax.bar(x-0.2, effs_cf, 0.38, label='Efficiency (%)', color="powderblue",   alpha=0.85, zorder=3)
ax.bar(x+0.2, purs_cf, 0.38, label='Purity (%)',     color="black", alpha=0.85, zorder=3)

for i, (e, p) in enumerate(zip(effs_cf, purs_cf)):
    ax.text(i-0.2, e+0.5, f'{e:.0f}', ha='center', va='bottom', fontsize=8, color="powderblue")
    ax.text(i+0.2, p+0.5, f'{p:.0f}', ha='center', va='bottom', fontsize=8, color="black")

ax.set_xticks(x)
ax.set_xticklabels(labels_cf, rotation=25, ha='right', fontsize=10)
ax.set_ylabel('Percentage (%)')
ax.set_title('Cut flow: efficiency and purity at each step')
ax.legend()
fig.tight_layout()
#savefig(fig, 'cutflow_bar.png')
plt.show()

## 5. Electron KE threshold sweep
---

Sweep the true electron KE threshold (0.075--0.5 GeV) used in the signal definition.
The Pareto map shows the efficiency-purity tradeoff coloured by threshold value.
The crimson dot marks the baseline (0.2 GeV) and the gold star marks the point
maximising eff $\times$ purity.

In [ ]:
KE_VALS   = np.array([0.075,0.100,0.125,0.150,0.175,0.200,
                       0.225,0.250,0.300,0.350,0.400,0.450,0.500])
BASE_KE   = 0.200
base_idx  = np.argmin(np.abs(KE_VALS-BASE_KE))

effs_ke, purs_ke, nsigs_ke = [], [], []
for ke in KE_VALS:
    sk   = signal_mask(df, ke)
    sel  = sel_full(df)
    n_s  = (sel & sk).sum()
    effs_ke.append(n_s/sk.sum()   if sk.sum()>0   else 0)
    purs_ke.append(n_s/sel.sum()  if sel.sum()>0  else 0)
    nsigs_ke.append(sk.sum())

effs_ke  = np.array(effs_ke)
purs_ke  = np.array(purs_ke)
nsigs_ke = np.array(nsigs_ke)
fom_ke   = effs_ke * purs_ke
opt_idx  = np.argmax(fom_ke)

# Binomial errors
eff_err = np.sqrt(effs_ke*(1-effs_ke)/np.maximum(nsigs_ke,1))
pur_err = np.sqrt(purs_ke*(1-purs_ke)/np.maximum(sel_full(df).sum(),1)) * np.ones_like(purs_ke)

fig = plt.figure(figsize=(17, 5), dpi=300)
gs  = fig.add_gridspec(1, 4, wspace=0.38)
axes = [fig.add_subplot(gs[i]) for i in range(4)]

for ax, vals, err, ylabel in [
        (axes[0], effs_ke,  eff_err, 'Efficiency'),
        (axes[1], purs_ke,  pur_err, 'Purity'),
        (axes[2], nsigs_ke, np.sqrt(nsigs_ke), 'N true signal events')]:
    band_plot(ax, KE_VALS, vals, err, BLUE, lw=2, ms=5)
    vline_base(ax, BASE_KE)
    vline_opt(ax, KE_VALS[opt_idx])
    ax.set_xlabel(r'True $e^{\pm}$ KE threshold [GeV]')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)

# Pareto map
ax3 = axes[3]
sc  = ax3.scatter(effs_ke, purs_ke, c=KE_VALS, cmap='plasma',
                  s=55, zorder=5, edgecolors='white', linewidths=0.4)
ax3.plot(effs_ke, purs_ke, '-', color='#aaaaaa', lw=1, zorder=3)
plt.colorbar(sc, ax=ax3, label=r'KE threshold [GeV]')

# error ellipses at baseline and optimum
for idx, color, marker, ms in [(base_idx, CRIMSON, 'o', 120),
                                 (opt_idx,  GOLD,   '*', 160)]:
    ax3.scatter(effs_ke[idx], purs_ke[idx],
                s=ms, color=color, marker=marker, zorder=7,
                edgecolors='white', linewidths=0.8)
    ax3.errorbar(effs_ke[idx], purs_ke[idx],
                 xerr=eff_err[idx], yerr=pur_err[idx],
                 fmt='none', color=color, lw=1.5, capsize=4, zorder=6)

# dashed crosshairs at baseline
ax3.axhline(purs_ke[base_idx], ls='--', lw=0.9, color=CRIMSON, alpha=0.4)
ax3.axvline(effs_ke[base_idx], ls='--', lw=0.9, color=CRIMSON, alpha=0.4)

from matplotlib.lines import Line2D
ax3.legend(handles=[
    Line2D([0],[0],marker='o',color='w',markerfacecolor=CRIMSON,ms=9,
           label=f'Baseline ({BASE_KE} GeV)'),
    Line2D([0],[0],marker='*',color='w',markerfacecolor=GOLD,ms=11,
           label=f'Opt. eff*pur ({KE_VALS[opt_idx]:.3f} GeV)'),
], fontsize=8)
ax3.set_xlabel('Efficiency')
ax3.set_ylabel('Purity')
ax3.set_title('Efficiency-purity Pareto frontier')

fig.suptitle(r'True $e^{\pm}$ KE threshold sweep (no trigger, Pandora)',
             fontsize=13, y=1.02)
#savefig(fig, 'ke_threshold_sweep.png')
plt.show()

print(f'Baseline ({BASE_KE} GeV): eff={effs_ke[base_idx]:.3f}, pur={purs_ke[base_idx]:.3f}')
print(f'Optimum  ({KE_VALS[opt_idx]:.3f} GeV): eff={effs_ke[opt_idx]:.3f}, pur={purs_ke[opt_idx]:.3f}')

## 6. Individual cut sweeps with Punzi FOM
---

Each cut is swept while all others are held at baseline.
**Punzi FOM** $= \varepsilon / (1 + \sqrt{N_{B}})$ is the standard figure of
merit for optimising a cut targeting 90% CL sensitivity in a counting experiment.
The gold dotted line marks the FOM-optimal threshold for each cut.

In [ ]:
def run_sweep(df, sig, vals, full_fn):
    n_tot = sig.sum()
    effs, purs, foms = [], [], []
    for v in vals:
        sel   = full_fn(df, v)
        n_sig = (sel & sig).sum()
        n_sel = sel.sum()
        n_bkg = n_sel - n_sig
        eff   = n_sig/n_tot if n_tot>0 else 0
        pur   = n_sig/n_sel if n_sel>0 else 0
        effs.append(eff)
        purs.append(pur)
        foms.append(eff/(1+np.sqrt(max(n_bkg,0))))
    e, p, f = np.array(effs), np.array(purs), np.array(foms)
    eff_err = np.sqrt(e*(1-e)/max(n_tot,1))
    pur_err = np.sqrt(p*(1-p)/np.maximum(np.array([sel_full(df).sum()]*len(vals)),1))
    return e, p, f, eff_err, pur_err

# full selection with one free parameter
def fs_e(df,v):   return sel_flash(df)    &c_shower(df,v)&c_dedx(df) &c_angle(df)&c_gap(df)&c_proton(df)&c_0pi(df)&c_muon(df)
def fs_dx(df,v):  return sel_shower(df)   &c_dedx(df,v)  &c_angle(df)&c_gap(df)  &c_proton(df)&c_0pi(df)&c_muon(df)
def fs_ang(df,v): return sel_dedx(df)     &c_angle(df,v) &c_gap(df)  &c_proton(df)&c_0pi(df)&c_muon(df)
def fs_gap(df,v): return sel_dedx(df)     &c_angle(df)   &c_gap(df,v)&c_proton(df)&c_0pi(df)&c_muon(df)
def fs_mu(df,v):  return sel_0pi(df)      &c_muon(df,v)

sweeps = [
    ('Shower energy min [GeV]',            np.linspace(0.05,0.5,35),  SHOWER_ENERGY_MIN,  fs_e),
    ('Shower dE/dx max [MeV/cm]',          np.linspace(0.8, 9.0,35),  SHOWER_DEDX_MAX,    fs_dx),
    ('Open angle max [deg]',               np.linspace(2.0,40.0,35),  SHOWER_ANGLE_MAX,   fs_ang),
    ('Conv. gap max [cm]',                 np.linspace(0.5,15.0,35),  SHOWER_CONVGAP_MAX, fs_gap),
    (r'$\chi^{2}_{\mu}$ veto max',          np.linspace(10.,100.,35), MUON_CHI2_MU_MAX,   fs_mu),
]

fig, axes = plt.subplots(3, len(sweeps), figsize=(20,11),
                          gridspec_kw={'hspace':0.50,'wspace':0.38}, dpi=300)

for col, (xlabel, vals, base, sfn) in enumerate(sweeps):
    effs, purs, foms, eff_err, pur_err = run_sweep(df, sig, vals, sfn)
    opt_v = vals[np.argmax(foms)]
    fom_err = np.abs(foms) * np.sqrt((eff_err/np.where(effs>0,effs,1))**2)

    for row, (y, err, ylabel) in enumerate([
            (effs, eff_err, 'Efficiency'),
            (purs, pur_err, 'Purity'),
            (foms, fom_err, 'Punzi FOM')]):
        ax = axes[row, col]
        band_plot(ax, vals, y, err, BLUE, lw=2, ms=0, fmt='')
        ax.plot(vals, y, '-', color=BLUE, lw=2, zorder=4)
        vline_base(ax, base)
        vline_opt(ax, opt_v)
        ax.set_xlabel(xlabel, fontsize=9)
        ax.set_ylabel(ylabel, fontsize=10)
        if row==0:
            ax.set_title(xlabel.split('[')[0].strip(), fontsize=9)
        if col==0 or row==2:
            ax.legend(fontsize=7)

fig.suptitle('Individual cut sweeps -- efficiency, purity, Punzi FOM\n'
             '(no trigger, Pandora-only, all others at baseline)',
             fontsize=12, y=1.01)
#savefig(fig, 'cut_sweeps.png')
plt.show()

## 7. Cut ordering study
---
Does the order in which various cuts are applied impact efficiencies and purities?

In [ ]:
ATOMIC = {
    'Presel.':   c_presel,  'FM':        c_flash,
    'Shower E':  c_shower,  'dE/dx':     c_dedx,
    'Angle':     c_angle,   'Gap':       c_gap,
    'Proton':    c_proton,  '$0\\pi$':    c_0pi,
    'Muon veto': c_muon,
}

def apply_ordering(df, sig, order):
    effs, purs = [], []
    mask = pd.Series(True, index=df.index)
    n_tot = sig.sum()
    for name in order:
        mask  = mask & ATOMIC[name](df)
        n_sig = (mask & sig).sum()
        n_sel = mask.sum()
        effs.append(n_sig/n_tot  if n_tot>0  else 0)
        purs.append(n_sig/n_sel  if n_sel>0  else 0)
    return np.array(effs), np.array(purs)

ORDER_STD  = ['Presel.','FM','Shower E','dE/dx','Angle','Gap','Proton','$0\\pi$','Muon veto']
ORDER_DEDX = ['Presel.','FM','dE/dx','Shower E','Angle','Gap','Proton','$0\\pi$','Muon veto']
ORDER_TOPO = ['Presel.','FM','Proton','$0\\pi$','Shower E','dE/dx','Angle','Gap','Muon veto']

ords = [
    ('Standard',       ORDER_STD,  '#4257f5'),
    ('dE/dx first',    ORDER_DEDX, '#f39c12'),
    ('Topology first', ORDER_TOPO, '#3f8f3f'),
]

fig, (ax_e, ax_p) = plt.subplots(1,2, figsize=(14,5.5), sharey=False)
x = np.arange(len(ORDER_STD))

for name, order, col in ords:
    effs, purs = apply_ordering(df, sig, order)
    eff_err = np.sqrt(effs*(1-effs)/max(sig.sum(),1))
    pur_err = np.sqrt(purs*(1-purs)/np.maximum(
        [sum(pd.Series(True,index=df.index)
             &pd.concat([ATOMIC[k](df) for k in order[:i+1]],axis=1).all(axis=1))
         for i in range(len(order))], 1))

    for ax, vals, err in [(ax_e, effs, eff_err),(ax_p, purs, pur_err)]:
        ax.fill_between(x, vals-err, vals+err, color=col, alpha=0.15, zorder=2)
        ax.plot(x, vals, 'o-', color=col, lw=2, ms=5.5, label=name,
                markeredgecolor='white', markeredgewidth=0.5, zorder=4)

xlabs = [s.replace('$','').replace('\\','') for s in ORDER_STD]
for ax, ylabel, title in [(ax_e,'Efficiency','Efficiency vs cut ordering'),
                           (ax_p,'Purity',    'Purity vs cut ordering')]:
    ax.set_xticks(list(x))
    ax.set_xticklabels(xlabs, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_ylim(0,1.05)
    ax.set_title(title)
    ax.legend()

fig.suptitle('Cut ordering study (no trigger, Pandora)', y=1.02)
fig.tight_layout()
#savefig(fig, 'cut_ordering.png')
plt.show()

## 8. 2D efficiency and purity maps (smoothed)
---

Examples --> trade-off in threshold placement of various cut variables can be visualised this way. 

In [ ]:
def smooth_nan(a, sigma=1.2):
    """NaN-aware Gaussian smoothing."""
    filled = np.where(np.isnan(a), 0., a)
    weight = np.where(np.isnan(a), 0., 1.)
    sf = gaussian_filter(filled, sigma)
    sw = gaussian_filter(weight, sigma)
    return np.where(sw > 0.1, sf/sw, np.nan)

def plot_2d(df, sig, vx, vy, lx, ly, bx, by, sel, title, fname,
            min_count=3, sigma=1.2):
    ok = df[vx].notna() & df[vy].notna()
    def h2(mask):
        return np.histogram2d(df.loc[ok&mask,vx], df.loc[ok&mask,vy],
                              bins=[bx,by])[0]
    den = h2(sig)
    num = h2(sig&sel)
    tot = h2(sel)

    eff = np.where(den>min_count, num/den, np.nan)
    pur = np.where(tot>min_count, num/tot, np.nan)
    eff_s = smooth_nan(eff, sigma)
    pur_s = smooth_nan(pur, sigma)

    fig, axes = plt.subplots(1,2, figsize=(14,5.5), dpi=300)
    for ax, data, clabel in zip(axes,[eff_s,pur_s],['Efficiency','Purity']):
        im = ax.pcolormesh(bx, by, data.T, cmap=MANAGUA, vmin=0, vmax=1)
        cb = plt.colorbar(im, ax=ax, label=clabel)
        cb.ax.tick_params(labelsize=10)
        ax.set_xlabel(lx)
        ax.set_ylabel(ly)
        ax.set_title(f'{clabel} -- {title}')

    fig.tight_layout()
    #savefig(fig, fname)
    plt.show()

sel = sel_full(df)

plot_2d(df, sig,
    'shower_energy','shower_avail_dedx',
    'Shower energy [GeV]','Shower dE/dx [MeV/cm]',
    np.linspace(0,2.5,30), np.linspace(0,10,30),
    sel, 'shower energy vs dE/dx', 'map_energy_dedx.png')

plot_2d(df, sig,
    'shower_energy','shower_open_angle',
    'Shower energy [GeV]','Open angle [deg]',
    np.linspace(0,2.5,30), np.linspace(0,40,30),
    sel, 'shower energy vs open angle', 'map_energy_angle.png')

# Proton momentum vs leading proton chi2
plot_2d(df, sig,
    'lead_proton_momentum','lead_proton_chi2_proton',
    'Leading proton momentum [GeV/c]',r'Proton $\chi^{2}_{p}$',
    np.linspace(0,1.5,28), np.linspace(0,200,28),
    sel, 'proton momentum vs chi2', 'map_proton_chi2.png')

## 9. Neutrino energy resolution and regression
---

Linear fit $E_{reco} = m \cdot E_{true} + c$ for selected signal events.
Slope < 1 indicates systematic underestimation (expected: missing neutron energy,
sub-threshold protons, calorimetric inefficiency).

Resolution $\sigma[(E_{reco}-E_{true})/E_{true}]$ is fitted with a linear
model $\sigma = a + b\cdot E_{true}$ to characterise the energy dependence.

In [ ]:
ok = (sel_full(df) & sig &
      df['reco_nu_energy'].notna() & df['true_nu_energy'].notna() &
      (df['reco_nu_energy']>0))
Et = df.loc[ok,'true_nu_energy'].values
Er = df.loc[ok,'reco_nu_energy'].values
res = (Er - Et)/Et

# linear fit with uncertainties
m, c, r, p, se = linregress(Et, Er)
Ef  = np.linspace(0.1, 3.5, 300)
Ep  = m*Ef + c
n, Em = len(Et), Et.mean()
sxx  = ((Et-Em)**2).sum()
band = se * np.sqrt(1/n + (Ef-Em)**2/sxx)   # prediction band

# binned residuals
RB  = np.linspace(0.2, 3.2, 10)
RC  = 0.5*(RB[:-1]+RB[1:])
r_mu, r_sig, r_err = [], [], []
for lo,hi in zip(RB[:-1],RB[1:]):
    rb = res[(Et>=lo)&(Et<hi)]
    if len(rb)>5:
        r_mu.append(rb.mean()); r_sig.append(rb.std())
        r_err.append(rb.std()/np.sqrt(len(rb)))
    else:
        r_mu.append(np.nan); r_sig.append(np.nan); r_err.append(np.nan)
r_mu  = np.array(r_mu)
r_sig = np.array(r_sig)
r_err = np.array(r_err)

# resolution fit sigma = a + b*E
ok_b = ~np.isnan(r_sig)
try:
    (a_res, b_res), cov_res = curve_fit(lambda x,a,b: a+b*x, RC[ok_b], r_sig[ok_b])
    perr_res = np.sqrt(np.diag(cov_res))
    fit_ok = True
except:
    fit_ok = False

# -- Figure: 2x2 panels --------------------------------------------------------
fig = plt.figure(figsize=(14,10))
gs  = fig.add_gridspec(2, 2, hspace=0.40, wspace=0.35)
ax_main = fig.add_subplot(gs[0,:])
ax_res  = fig.add_subplot(gs[1,0])
ax_sig  = fig.add_subplot(gs[1,1])

# -- Main: 2D + fit ------------------------------------------------------------
h,xb,yb,im = ax_main.hist2d(Et, Er, bins=32, range=[[0,3],[0,3]], cmap=MANAGUA)
plt.colorbar(im, ax=ax_main, label='Events')

ax_main.plot(Ef, Ef, 'w--', lw=1.8, label='Perfect reconstruction', zorder=5)
ax_main.plot(Ef, Ep, color='#ff6b6b', lw=2.5, zorder=6,
             label=fr'Fit: $E_{{reco}} = {m:.3f}\,E_{{true}} {c:+.3f}$ ($R^2={r**2:.3f}$)')
ax_main.fill_between(Ef, Ep-band, Ep+band,
                      color='#ff6b6b', alpha=0.22, zorder=4,
                      label=r'$1\sigma$ fit uncertainty')

ax_main.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_main.set_ylabel(r'Reco $E_{\nu}$ [GeV]')
ax_main.set_title(r'Neutrino energy: reco vs true (selected signal, no trigger, Pandora)')
ax_main.legend(fontsize=9, loc='upper left')
ax_main.set_xlim(0,3); ax_main.set_ylim(0,3)

# -- Bottom left: mean residual ------------------------------------------------
ax_res.axhline(0, ls='--', lw=1.5, color='#555555', zorder=5)
ax_res.fill_between(RC, r_mu-r_sig, r_mu+r_sig,
                     color=BLUE, alpha=0.15, step='mid', zorder=2,
                     label=r'$\pm 1\sigma$ spread')
ax_res.errorbar(RC, r_mu, yerr=r_err,
                fmt='o', ms=6, lw=1.8, capsize=4, color=BLUE,
                markeredgecolor='white', markeredgewidth=0.5,
                label='Mean residual', zorder=5)
ax_res.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_res.set_ylabel(r'$(E_{reco}-E_{true})/E_{true}$')
ax_res.set_title('Mean fractional residual vs energy')
ax_res.legend(fontsize=9)

# -- Bottom right: resolution (sigma) -----------------------------------------
if fit_ok:
    ax_sig.plot(Ef, a_res+b_res*Ef, color='#ff6b6b', lw=2.2, zorder=4,
                label=fr'Fit: $\sigma={a_res:.3f}\pm{perr_res[0]:.3f}$ '
                      fr'$+({b_res:.3f}\pm{perr_res[1]:.3f})\,E_{{true}}$')
    ax_sig.fill_between(Ef,
        (a_res-perr_res[0])+(b_res-perr_res[1])*Ef,
        (a_res+perr_res[0])+(b_res+perr_res[1])*Ef,
        color='#ff6b6b', alpha=0.18, zorder=3)
    print(f'Resolution: sigma = {a_res:.4f}(+-{perr_res[0]:.4f}) + {b_res:.4f}(+-{perr_res[1]:.4f}) * E_true')

ax_sig.fill_between(RC[ok_b], (r_sig-r_err)[ok_b], (r_sig+r_err)[ok_b],
                    color=BLUE, alpha=0.18, step='mid', zorder=2)
ax_sig.plot(RC[ok_b], r_sig[ok_b], 'o', color=BLUE, ms=6, zorder=5,
            markeredgecolor='white', markeredgewidth=0.5, label='Resolution')
ax_sig.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_sig.set_ylabel(r'$\sigma[(E_{reco}-E_{true})/E_{true}]$')
ax_sig.set_title('Energy resolution (RMS) vs energy')
ax_sig.set_ylim(bottom=0)
ax_sig.legend(fontsize=8)

print(f'Linear fit: E_reco = {m:.4f} * E_true {c:+.4f}  (R^2={r**2:.4f})')
print(f'Slope < 1 ({m:.4f}) => systematic underestimation of E_nu')

#savefig(fig, 'energy_resolution.png')
plt.show()

## Energy Resolution: Fitting Method Notes
---
### The problem
Each vertical slice of the reco vs true E_nu 2D map has an asymmetric
distribution -- a sharp rising edge near the peak and a long tail dragging to lower reco energies. This tail comes from events where significant energy is lost through mechanisms that are not visible: neutrons, sub-threshold protons, and hadronic activity below the detection threshold. A naive mean of this distribution is pulled far below the physical peak, making the fit appear worse than it is. The goal is to extract the **most probable value (MPV)** -- the peak of the distribution -- not the mean.


In [ ]:
from scipy.stats import norm as sp_norm

# redefine signal mask inline to avoid any variable clash
sig_mask = (df['is_signal']==1) & (df['true_electron_ke'].fillna(0) > 0.2)

ok = (np.array(sel_full(df), dtype=bool) &
      np.array(sig_mask, dtype=bool) &
      np.array(df['reco_nu_energy'].notna(), dtype=bool) &
      np.array(df['true_nu_energy'].notna(), dtype=bool) &
      np.array(df['reco_nu_energy'].values > 0, dtype=bool))

Et  = df['true_nu_energy'].values[ok]
Er  = df['reco_nu_energy'].values[ok]
res = (Er - Et) / Et

print(f'N events: {ok.sum()}')
print(f'Mean residual: {res.mean():.3f}')
print(f'Et range: {Et.min():.2f} - {Et.max():.2f} GeV')
print(f'Er range: {Er.min():.2f} - {Er.max():.2f} GeV')

### Method 1: Symmetric Gaussian 
A standard Gaussian fit to each E_true slice finds the mean and width.
Since the distribution is asymmetric, the mean sits significantly below
the peak -- the long low-energy tail pulls it down. The resulting fit line lies well below the data ridge, and residuals are systematically negative with no way to be symmetric around zero. The Gaussian is the wrong functional form for this distribution.

In [ ]:
# redefine signal mask inline to avoid any variable clash
sig_mask = (df['is_signal']==1) & (df['true_electron_ke'].fillna(0) > 0.2)

ok = (np.array(sel_full(df), dtype=bool) &
      np.array(sig_mask, dtype=bool) &
      np.array(df['reco_nu_energy'].notna(), dtype=bool) &
      np.array(df['true_nu_energy'].notna(), dtype=bool) &
      np.array(df['reco_nu_energy'].values > 0, dtype=bool))

Et  = df['true_nu_energy'].values[ok]
Er  = df['reco_nu_energy'].values[ok]
res = (Er - Et) / Et

print(f'N events: {ok.sum()}')
print(f'Mean residual: {res.mean():.3f}')

# Gaussian peak per E_true bin
FIT_LO, FIT_HI = 0.3, 2.2
E_TRUE_BINS = np.linspace(FIT_LO, FIT_HI, 12)
E_TRUE_CEN  = 0.5*(E_TRUE_BINS[:-1]+E_TRUE_BINS[1:])

peak_reco, peak_err = [], []
for lo, hi in zip(E_TRUE_BINS[:-1], E_TRUE_BINS[1:]):
    er_bin = Er[(Et>=lo) & (Et<hi)]
    if len(er_bin) > 10:
        try:
            mu, sg = sp_norm.fit(er_bin)   # sg not sig!
            peak_reco.append(mu)
            peak_err.append(sg/np.sqrt(len(er_bin)))
        except:
            peak_reco.append(np.nan); peak_err.append(np.nan)
    else:
        peak_reco.append(np.nan); peak_err.append(np.nan)

peak_reco = np.array(peak_reco)
peak_err  = np.array(peak_err)
ok_p      = ~np.isnan(peak_reco)

# linear fit through Gaussian peaks
m, c, r_val, _, se = linregress(E_TRUE_CEN[ok_p], peak_reco[ok_p])
Ef    = np.linspace(FIT_LO, FIT_HI, 300)
Ep    = m*Ef + c
n_fit = ok_p.sum()
Em    = E_TRUE_CEN[ok_p].mean()
sxx   = ((E_TRUE_CEN[ok_p]-Em)**2).sum()
band  = se * np.sqrt(1/n_fit + (Ef-Em)**2/sxx)

print(f'Ridge fit (Gaussian peaks, {FIT_LO}-{FIT_HI} GeV):')
print(f'  slope     = {m:.4f}')
print(f'  intercept = {c:.4f} GeV')
print(f'  R^2       = {r_val**2:.4f}')

# Binned residuals 
RB = np.linspace(0.2, 3.2, 10)
RC = 0.5*(RB[:-1]+RB[1:])
r_mu, r_sg, r_err = [], [], []
for lo, hi in zip(RB[:-1], RB[1:]):
    rb = res[(Et>=lo) & (Et<hi)]
    if len(rb) > 5:
        r_mu.append(rb.mean())
        r_sg.append(rb.std())
        r_err.append(rb.std()/np.sqrt(len(rb)))
    else:
        r_mu.append(np.nan); r_sg.append(np.nan); r_err.append(np.nan)

r_mu  = np.array(r_mu)
r_sg  = np.array(r_sg)
r_err = np.array(r_err)

# Resolution fit sigma = a + b*E 
ok_b = ~np.isnan(r_sg)
try:
    (a_res,b_res), cov_res = curve_fit(lambda x,a,b: a+b*x, RC[ok_b], r_sg[ok_b])
    perr_res = np.sqrt(np.diag(cov_res))
    fit_ok = True
    print(f'  Resolution: sigma = {a_res:.3f}+-{perr_res[0]:.3f} + ({b_res:.3f}+-{perr_res[1]:.3f})*E_true')
except:
    fit_ok = False

# Figure 
fig = plt.figure(figsize=(14,10))
gs  = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.35)
ax_main = fig.add_subplot(gs[0,:])
ax_res  = fig.add_subplot(gs[1,0])
ax_sg   = fig.add_subplot(gs[1,1])

# Main: 2D map + ridge fit 
h,xb,yb,im = ax_main.hist2d(Et, Er, bins=32, range=[[0,3],[0,3]], cmap=MANAGUA)
plt.colorbar(im, ax=ax_main, label='Events')

ax_main.plot([0,3],[0,3], 'w--', lw=1.8, label='Perfect reconstruction', zorder=5)
ax_main.errorbar(E_TRUE_CEN[ok_p], peak_reco[ok_p], yerr=peak_err[ok_p],
                 fmt='o', ms=5, color='white', markeredgecolor='#ff6b6b',
                 markeredgewidth=1.5, capsize=3, lw=1.2, zorder=7,
                 label='Gaussian peak per bin')
ax_main.plot(Ef, Ep, color='#ff6b6b', lw=2.5, zorder=6,
             label=fr'Ridge fit: $E_{{reco}}={m:.3f}\,E_{{true}}{c:+.3f}$ ($R^2={r_val**2:.3f}$)')
ax_main.fill_between(Ef, Ep-band, Ep+band,
                     color='#ff6b6b', alpha=0.22, zorder=4, label=r'$1\sigma$ fit band')
ax_main.axvline(FIT_LO, ls=':', lw=1, color='#ff6b6b', alpha=0.4)
ax_main.axvline(FIT_HI, ls=':', lw=1, color='#ff6b6b', alpha=0.4)

ax_main.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_main.set_ylabel(r'Reco $E_{\nu}$ [GeV]')
ax_main.set_title(r'Neutrino energy: reco vs true (selected signal, no trigger, Pandora)')
ax_main.legend(fontsize=9, loc='upper left')
ax_main.set_xlim(0,3); ax_main.set_ylim(0,3)

# Bottom left: mean residual
ax_res.axhline(0, ls='--', lw=1.5, color='#555555', zorder=5)
ax_res.fill_between(RC, r_mu-r_sg, r_mu+r_sg,
                    color=BLUE, alpha=0.15, step='mid', zorder=2,
                    label=r'$\pm 1\sigma$ spread')
ax_res.errorbar(RC, r_mu, yerr=r_err,
                fmt='o', ms=6, lw=1.8, capsize=4, color=BLUE,
                markeredgecolor='white', markeredgewidth=0.5,
                label='Mean residual', zorder=5)
ax_res.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_res.set_ylabel(r'$(E_{reco}-E_{true})/E_{true}$')
ax_res.set_title('Mean fractional residual vs energy')
ax_res.legend(fontsize=9)

# Bottom right: resolution 
Ef_res = np.linspace(0.2, 3.2, 200)
if fit_ok:
    ax_sg.plot(Ef_res, a_res+b_res*Ef_res, color='#ff6b6b', lw=2.2, zorder=4,
               label=fr'Fit: $\sigma={a_res:.3f}\pm{perr_res[0]:.3f}'
                     fr'+({b_res:.3f}\pm{perr_res[1]:.3f})\,E_{{true}}$')
    ax_sg.fill_between(Ef_res,
        np.maximum(0,(a_res-perr_res[0])+(b_res-perr_res[1])*Ef_res),
        (a_res+perr_res[0])+(b_res+perr_res[1])*Ef_res,
        color='#ff6b6b', alpha=0.18, zorder=3)

ax_sg.fill_between(RC[ok_b], np.maximum(0,(r_sg-r_err)[ok_b]), (r_sg+r_err)[ok_b],
                   color=BLUE, alpha=0.18, step='mid', zorder=2)
ax_sg.plot(RC[ok_b], r_sg[ok_b], 'o', color=BLUE, ms=6, zorder=5,
           markeredgecolor='white', markeredgewidth=0.5, label='Resolution')
ax_sg.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_sg.set_ylabel(r'$\sigma[(E_{reco}-E_{true})/E_{true}]$')
ax_sg.set_title('Energy resolution (RMS) vs energy')
ax_sg.set_ylim(bottom=0)
ax_sg.legend(fontsize=8)

#savefig(fig, 'energy_resolution.png')
plt.show()

### Method 2: KDE peak (kernel density estimate)
Instead of fitting a parametric function, a KDE smooths the data with a
Gaussian kernel and finds the x-value of the maximum density. This is
non-parametric and makes no assumption about the shape. It works better
than a symmetric Gaussian because it can find the true mode of an
asymmetric distribution. However, it is sensitive to the bandwidth
parameter: too wide and the peak is smeared, too narrow and statistical
noise dominates. With ~30 events per bin the KDE peak estimate is noisy
and jumps around between bins, giving inconsistent MPVs and residuals
that do not sit stably around zero.

In [ ]:
# Signal mask (inline to avoid variable clash) 
sig_mask = (df['is_signal']==1) & (df['true_electron_ke'].fillna(0) > 0.2)

ok = (np.array(sel_full(df), dtype=bool) &
      np.array(sig_mask, dtype=bool) &
      np.array(df['reco_nu_energy'].notna(), dtype=bool) &
      np.array(df['true_nu_energy'].notna(), dtype=bool) &
      np.array(df['reco_nu_energy'].values > 0, dtype=bool))

Et  = df['true_nu_energy'].values[ok]
Er  = df['reco_nu_energy'].values[ok]

print(f'N selected signal: {ok.sum()}')

# E_true bins for slice fits 
FIT_LO, FIT_HI = 0.3, 2.2
E_TRUE_BINS = np.linspace(FIT_LO, FIT_HI, 12)
E_TRUE_CEN  = 0.5*(E_TRUE_BINS[:-1]+E_TRUE_BINS[1:])
N_BINS      = len(E_TRUE_CEN)
N_RECO_BINS = 18

# Per-slice MPV (windowed histogram mode) 
peak_reco, peak_err = [], []
slice_data   = []
slice_counts = []
slice_edges  = []

from scipy.stats import gaussian_kde

# replace the inner loop with:
for lo, hi in zip(E_TRUE_BINS[:-1], E_TRUE_BINS[1:]):
    er_bin = Er[(Et>=lo) & (Et<hi)]
    slice_data.append(er_bin)
    if len(er_bin) > 8:
        expected = 0.5*(lo+hi)
        win_lo   = max(0,   expected - 0.6)
        win_hi   = min(3.5, expected + 0.6)
        
        # KDE peak finder
        er_win = er_bin[(er_bin>=win_lo) & (er_bin<=win_hi)]
        if len(er_win) > 5:
            kde      = gaussian_kde(er_win, bw_method=0.3)
            x_scan   = np.linspace(win_lo, win_hi, 200)
            mpv      = x_scan[np.argmax(kde(x_scan))]
            # bootstrap uncertainty on MPV
            boots = [x_scan[np.argmax(gaussian_kde(
                         er_win[np.random.choice(len(er_win), len(er_win))],
                         bw_method=0.3)(x_scan))]
                     for _ in range(50)]
            mpv_err = np.std(boots)
        else:
            mpv, mpv_err = np.nan, np.nan
            
        # still store histogram for the plot
        counts, edges = np.histogram(er_bin, bins=N_RECO_BINS,
                                     range=(win_lo, win_hi))
        slice_counts.append(counts); slice_edges.append(edges)
        peak_reco.append(mpv); peak_err.append(mpv_err)
    else:
        slice_counts.append(None); slice_edges.append(None)
        peak_reco.append(np.nan); peak_err.append(np.nan)

peak_reco = np.array(peak_reco)
peak_err  = np.array(peak_err)
ok_p      = ~np.isnan(peak_reco)

# Weighted linear fit through MPV peaks 
weights  = 1.0 / np.where(peak_err[ok_p]>0, peak_err[ok_p]**2, 1e6)
weights /= weights.sum()

coeffs, cov_p = np.polyfit(E_TRUE_CEN[ok_p], peak_reco[ok_p], 1,
                            w=weights, cov=True)
m, c   = coeffs
m_err  = np.sqrt(cov_p[0,0])
c_err  = np.sqrt(cov_p[1,1])

y_pred   = m*E_TRUE_CEN[ok_p] + c
ss_res   = np.sum(weights*(peak_reco[ok_p]-y_pred)**2)
ss_tot   = np.sum(weights*(peak_reco[ok_p] -
                            np.average(peak_reco[ok_p],weights=weights))**2)
r_val_sq = 1 - ss_res/ss_tot

Ef   = np.linspace(FIT_LO, FIT_HI, 300)
Ep   = m*Ef + c
band = np.sqrt((Ef*m_err)**2 + c_err**2)

print(f'Weighted MPV fit: E_reco = {m:.4f}+-{m_err:.4f} * E_true {c:+.4f}+-{c_err:.4f}')
print(f'Weighted R^2 = {r_val_sq:.4f}')

#  MPV-based residuals (consistent with ridge fit) 
r_mu  = np.where(ok_p, (peak_reco - E_TRUE_CEN) / E_TRUE_CEN, np.nan)
r_err = np.where(ok_p, peak_err / E_TRUE_CEN, np.nan)

# spread = RMS of (Er - ridge_predicted)/Et in each bin
r_sg = []
for i, (lo, hi) in enumerate(zip(E_TRUE_BINS[:-1], E_TRUE_BINS[1:])):
    mask_bin = (Et>=lo) & (Et<hi)
    er_bin   = Er[mask_bin]
    et_bin   = Et[mask_bin]
    if ok_p[i] and len(er_bin) > 5:
        predicted = m * E_TRUE_CEN[i] + c
        spread    = (er_bin - predicted) / np.where(et_bin>0, et_bin, 1)
        r_sg.append(np.std(spread))
    else:
        r_sg.append(np.nan)
r_sg = np.array(r_sg)

# Resolution fit sigma = a + b*E 
ok_b = ok_p & ~np.isnan(r_sg)
try:
    (a_res,b_res), cov_res = curve_fit(lambda x,a,b: a+b*x,
                                        E_TRUE_CEN[ok_b], r_sg[ok_b])
    perr_res = np.sqrt(np.diag(cov_res))
    fit_ok   = True
    print(f'Resolution: sigma = {a_res:.3f}+-{perr_res[0]:.3f} + ({b_res:.3f}+-{perr_res[1]:.3f})*E_true')
except:
    fit_ok = False

# =============================================================================
# PLOT 1: per-slice diagnostic
# =============================================================================
ncols = 4
nrows = int(np.ceil(N_BINS / ncols))
fig_s, axes_s = plt.subplots(nrows, ncols, figsize=(14, 3.2*nrows),
                              gridspec_kw={'hspace':0.55,'wspace':0.35})
axes_s = axes_s.flatten()

for i, (lo, hi) in enumerate(zip(E_TRUE_BINS[:-1], E_TRUE_BINS[1:])):
    ax    = axes_s[i]
    er_b  = slice_data[i]
    label = fr'$E_{{true}}\in[{lo:.2f},{hi:.2f}]$ GeV'

    if slice_counts[i] is not None and len(er_b) > 8:
        counts = slice_counts[i]
        edges  = slice_edges[i]
        cens   = 0.5*(edges[:-1]+edges[1:])
        width  = edges[1]-edges[0]
        errs   = np.sqrt(np.maximum(counts, 1))

        ax.bar(cens, counts, width=width*0.85,
               color=BLUE, alpha=0.55, zorder=2)
        ax.errorbar(cens, counts, yerr=errs, fmt='none',
                    color=BLUE, lw=1.2, capsize=2, zorder=3)

        if ok_p[i]:
            mpv = peak_reco[i]
            ax.axvline(mpv, color=CRIMSON, lw=2, ls='--', zorder=4,
                       label=f'MPV={mpv:.2f} GeV')
            ax.axvspan(mpv-peak_err[i], mpv+peak_err[i],
                       color=CRIMSON, alpha=0.15, zorder=3)
            # also show ridge prediction
            ridge_pred = m*E_TRUE_CEN[i] + c
            ax.axvline(ridge_pred, color=GOLD, lw=1.5, ls=':', zorder=4,
                       label=f'Ridge={ridge_pred:.2f} GeV')
            ax.legend(fontsize=7, loc='upper right')

        ax.text(0.04, 0.96, f'N={len(er_b)}', transform=ax.transAxes,
                fontsize=7, va='top', color='#444444')
    else:
        ax.text(0.5, 0.5, f'{label}\nInsufficient stats\n(N={len(er_b)})',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=8, color='#888888')

    ax.set_title(label, fontsize=8)
    ax.set_xlabel(r'Reco $E_{\nu}$ [GeV]', fontsize=8)
    ax.set_ylabel('Events', fontsize=8)
    ax.tick_params(labelsize=7)

for j in range(i+1, len(axes_s)):
    axes_s[j].set_visible(False)

fig_s.suptitle(
    r'Reco $E_{\nu}$ distributions per true $E_{\nu}$ slice'
    '\n(crimson = MPV, gold = ridge prediction)',
    fontsize=11)
savefig(fig_s, 'energy_slices.png')
plt.show()

# =============================================================================
# PLOT 2: resolution figure
# =============================================================================
fig = plt.figure(figsize=(14,10))
gs  = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.35)
ax_main = fig.add_subplot(gs[0,:])
ax_res  = fig.add_subplot(gs[1,0])
ax_sg   = fig.add_subplot(gs[1,1])

# Main panel 
h,xb,yb,im = ax_main.hist2d(Et, Er, bins=32, range=[[0,3],[0,3]], cmap=MANAGUA)
plt.colorbar(im, ax=ax_main, label='Events')

ax_main.plot([0,3],[0,3], 'w--', lw=1.8, label='Perfect reconstruction', zorder=5)
ax_main.errorbar(E_TRUE_CEN[ok_p], peak_reco[ok_p], yerr=peak_err[ok_p],
                 fmt='o', ms=5.5, color='white', markeredgecolor=CRIMSON,
                 markeredgewidth=1.8, capsize=3, lw=1.4, zorder=7,
                 label='MPV per slice')
ax_main.plot(Ef, Ep, color=CRIMSON, lw=2.5, zorder=6,
             label=fr'Weighted MPV fit: '
                   fr'$E_{{reco}}={m:.3f}\pm{m_err:.3f}\,E_{{true}}{c:+.3f}\pm{c_err:.3f}$'
                   fr'  ($R^2={r_val_sq:.3f}$)')
ax_main.fill_between(Ef, Ep-band, Ep+band, color=CRIMSON, alpha=0.20,
                     zorder=4, label=r'$1\sigma$ fit uncertainty')
ax_main.axvline(FIT_LO, ls=':', lw=1, color=CRIMSON, alpha=0.4)
ax_main.axvline(FIT_HI, ls=':', lw=1, color=CRIMSON, alpha=0.4)
ax_main.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_main.set_ylabel(r'Reco $E_{\nu}$ [GeV]')
ax_main.set_title(r'Neutrino energy: reco vs true (selected signal, no trigger, Pandora)')
ax_main.legend(fontsize=8.5, loc='upper left')
ax_main.set_xlim(0,3); ax_main.set_ylim(0,3)

# MPV residual panel 
ax_res.axhline(0, ls='--', lw=1.5, color='#555555', zorder=5)
ax_res.fill_between(E_TRUE_CEN[ok_b],
                    (r_mu-r_sg)[ok_b], (r_mu+r_sg)[ok_b],
                    color=BLUE, alpha=0.15, step='mid', zorder=2,
                    label=r'$\pm 1\sigma$ spread (RMS around ridge)')
ax_res.errorbar(E_TRUE_CEN[ok_p], r_mu[ok_p], yerr=r_err[ok_p],
                fmt='o', ms=6, lw=1.8, capsize=4, color=BLUE,
                markeredgecolor='white', markeredgewidth=0.5,
                label='MPV residual', zorder=5)
ax_res.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_res.set_ylabel(r'$(E_{MPV}-E_{true})/E_{true}$')
ax_res.set_title('MPV fractional residual vs energy')
ax_res.legend(fontsize=9)

# Resolution panel 
Ef_res = np.linspace(FIT_LO, FIT_HI, 200)
if fit_ok:
    ax_sg.plot(Ef_res, a_res+b_res*Ef_res, color=CRIMSON, lw=2.2, zorder=4,
               label=fr'Fit: $\sigma={a_res:.3f}\pm{perr_res[0]:.3f}'
                     fr'+({b_res:.3f}\pm{perr_res[1]:.3f})\,E_{{true}}$')
    ax_sg.fill_between(Ef_res,
        np.maximum(0,(a_res-perr_res[0])+(b_res-perr_res[1])*Ef_res),
        (a_res+perr_res[0])+(b_res+perr_res[1])*Ef_res,
        color=CRIMSON, alpha=0.18, zorder=3)

ax_sg.fill_between(E_TRUE_CEN[ok_b],
                   np.maximum(0,(r_sg-r_err)[ok_b]), (r_sg+r_err)[ok_b],
                   color=BLUE, alpha=0.18, step='mid', zorder=2)
ax_sg.plot(E_TRUE_CEN[ok_b], r_sg[ok_b], 'o', color=BLUE, ms=6, zorder=5,
           markeredgecolor='white', markeredgewidth=0.5,
           label='Resolution (RMS around ridge)')
ax_sg.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_sg.set_ylabel(r'$\sigma[(E_{reco}-E_{ridge})/E_{true}]$')
ax_sg.set_title('Energy resolution (RMS around ridge) vs energy')
ax_sg.set_ylim(bottom=0)
ax_sg.legend(fontsize=8)

#savefig(fig, 'energy_resolution.png')
plt.show()

### Method 3: Bifurcated Gaussian
A bifurcated Gaussian has a shared peak mu but independent widths
sigma_left and sigma_right on either side. This is the natural model for a distribution with a physical peak and an asymmetric tail: sigma_right captures the narrow rising edge of the distribution (well-reconstructed events near the true energy), while sigma_left captures the broader falling tail (events with significant energy loss). The MPV is simply mu from the fit, independent of the tail shape. This is more stable than the KDE because it has a well-defined optimisation target (chi-squared minimisation) and propagates a proper uncertainty on mu from the covariance matrix. The residuals become much more symmetric around zero because the fit is no longer contaminated by the tail. The remaining scatter is genuine statistical noise from the finite sample size (~30 events per bin), not a systematic bias in the method.

In [ ]:
from scipy.optimize import curve_fit
from scipy.stats import gaussian_kde

# -- Signal mask (inline to avoid variable clash) ------------------------------
sig_mask = (df['is_signal']==1) & (df['true_electron_ke'].fillna(0) > 0.2)

ok = (np.array(sel_full(df), dtype=bool) &
      np.array(sig_mask, dtype=bool) &
      np.array(df['reco_nu_energy'].notna(), dtype=bool) &
      np.array(df['true_nu_energy'].notna(), dtype=bool) &
      np.array(df['reco_nu_energy'].values > 0, dtype=bool))

Et  = df['true_nu_energy'].values[ok]
Er  = df['reco_nu_energy'].values[ok]

print(f'N selected signal: {ok.sum()}')

# -- Bifurcated Gaussian -------------------------------------------------------
def bifurcated_gaussian(x, mu, sig_l, sig_r, norm):
    return np.where(x < mu,
                    norm * np.exp(-0.5*((x-mu)/sig_l)**2),
                    norm * np.exp(-0.5*((x-mu)/sig_r)**2))

def fit_bifurcated(er_win, win_lo, win_hi, n_bins=14):
    """
    Fit bifurcated Gaussian. Returns (mu, mu_err, popt).
    Sanity-checks mu stays within the physical window.
    Falls back to KDE if fit fails or result is unphysical.
    """
    counts, edges = np.histogram(er_win, bins=n_bins,
                                 range=(win_lo, win_hi))
    cens  = 0.5*(edges[:-1]+edges[1:])
    errs  = np.maximum(np.sqrt(counts), 1.)
    mu0   = cens[np.argmax(counts)]
    sg0   = np.std(er_win) * 0.5
    norm0 = float(counts.max())

    fit_ok = False
    try:
        popt, pcov = curve_fit(
            bifurcated_gaussian, cens, counts,
            p0    = [mu0, sg0, sg0*1.5, norm0],
            sigma = errs, absolute_sigma=True,
            bounds=([win_lo, 0.01, 0.01, 0.],
                    [win_hi,  1.5,  1.5, 1e6]),
            maxfev=8000)
        mu_err = np.sqrt(max(pcov[0,0], 0))
        # sanity: mu must be inside the window and mu_err reasonable
        if (win_lo < popt[0] < win_hi) and (0 < mu_err < 0.5):
            return popt[0], mu_err, popt
    except Exception:
        pass

    # KDE fallback
    try:
        x_sc  = np.linspace(win_lo, win_hi, 300)
        kde   = gaussian_kde(er_win[(er_win>=win_lo)&(er_win<=win_hi)],
                             bw_method=0.3)
        mpv   = x_sc[np.argmax(kde(x_sc))]
        rng   = np.random.default_rng(42)
        boots = []
        for _ in range(80):
            samp = er_win[rng.integers(0, len(er_win), len(er_win))]
            samp = samp[(samp>=win_lo)&(samp<=win_hi)]
            if len(samp) > 3:
                k = gaussian_kde(samp, bw_method=0.3)
                boots.append(x_sc[np.argmax(k(x_sc))])
        mu_err = np.std(boots) if boots else 0.1
        if win_lo < mpv < win_hi:
            return mpv, mu_err, None
    except Exception:
        pass

    return np.nan, np.nan, None

# -- E_true bins ---------------------------------------------------------------
FIT_LO, FIT_HI = 0.3, 2.2
E_TRUE_BINS = np.linspace(FIT_LO, FIT_HI, 12)
E_TRUE_CEN  = 0.5*(E_TRUE_BINS[:-1]+E_TRUE_BINS[1:])
N_BINS      = len(E_TRUE_CEN)
N_RECO_BINS = 16

peak_reco, peak_err = [], []
slice_data   = []
slice_counts = []
slice_edges  = []
slice_popt   = []

for lo, hi in zip(E_TRUE_BINS[:-1], E_TRUE_BINS[1:]):
    er_bin = Er[(Et>=lo) & (Et<hi)]
    slice_data.append(er_bin)
    if len(er_bin) > 8:
        expected = 0.5*(lo+hi)
        win_lo   = max(0.05, expected - 0.65)
        win_hi   = min(3.50, expected + 0.65)
        er_win   = er_bin[(er_bin>=win_lo) & (er_bin<=win_hi)]

        counts, edges = np.histogram(er_bin, bins=N_RECO_BINS,
                                     range=(win_lo, win_hi))
        slice_counts.append(counts)
        slice_edges.append(edges)

        if len(er_win) > 8:
            mpv, mpv_err, popt = fit_bifurcated(er_win, win_lo, win_hi,
                                                 n_bins=N_RECO_BINS)
        else:
            mpv, mpv_err, popt = np.nan, np.nan, None

        slice_popt.append(popt)
        peak_reco.append(mpv)
        peak_err.append(mpv_err)
    else:
        slice_counts.append(None); slice_edges.append(None)
        slice_popt.append(None)
        peak_reco.append(np.nan); peak_err.append(np.nan)

peak_reco = np.array(peak_reco)
peak_err  = np.array(peak_err)
ok_p      = ~np.isnan(peak_reco)

print(f'MPV found in {ok_p.sum()}/{N_BINS} bins')
for i in range(N_BINS):
    lo, hi = E_TRUE_BINS[i], E_TRUE_BINS[i+1]
    status = f'MPV={peak_reco[i]:.3f}+-{peak_err[i]:.3f}' if ok_p[i] else 'FAILED'
    method = 'bifurc.' if slice_popt[i] is not None else 'KDE'
    print(f'  [{lo:.2f},{hi:.2f}]: {status}  ({method})')

# -- Weighted linear fit -------------------------------------------------------
weights  = 1.0 / np.where(peak_err[ok_p]>0, peak_err[ok_p]**2, 1e6)
weights /= weights.sum()

coeffs, cov_p = np.polyfit(E_TRUE_CEN[ok_p], peak_reco[ok_p], 1,
                            w=weights, cov=True)
m, c   = coeffs
m_err  = np.sqrt(cov_p[0,0])
c_err  = np.sqrt(cov_p[1,1])

y_pred   = m*E_TRUE_CEN[ok_p] + c
ss_res   = np.sum(weights*(peak_reco[ok_p]-y_pred)**2)
ss_tot   = np.sum(weights*(peak_reco[ok_p] -
                            np.average(peak_reco[ok_p],weights=weights))**2)
r_val_sq = 1 - ss_res/ss_tot

Ef   = np.linspace(FIT_LO, FIT_HI, 300)
Ep   = m*Ef + c
band = np.sqrt((Ef*m_err)**2 + c_err**2)

print(f'\nRidge fit: E_reco = {m:.4f}+-{m_err:.4f} * E_true {c:+.4f}+-{c_err:.4f}')
print(f'Weighted R^2 = {r_val_sq:.4f}')

# -- MPV-based residuals -------------------------------------------------------
r_mu  = np.where(ok_p, (peak_reco - E_TRUE_CEN) / E_TRUE_CEN, np.nan)
r_err = np.where(ok_p, peak_err / E_TRUE_CEN, np.nan)

r_sg = []
for i, (lo, hi) in enumerate(zip(E_TRUE_BINS[:-1], E_TRUE_BINS[1:])):
    mask_bin   = (Et>=lo) & (Et<hi)
    er_b, et_b = Er[mask_bin], Et[mask_bin]
    if ok_p[i] and len(er_b) > 5:
        pred   = m*E_TRUE_CEN[i] + c
        spread = (er_b - pred) / np.where(et_b>0, et_b, 1)
        r_sg.append(np.std(spread))
    else:
        r_sg.append(np.nan)
r_sg = np.array(r_sg)

# -- Resolution fit ------------------------------------------------------------
ok_b = ok_p & ~np.isnan(r_sg)
try:
    (a_res,b_res), cov_res = curve_fit(lambda x,a,b: a+b*x,
                                        E_TRUE_CEN[ok_b], r_sg[ok_b])
    perr_res = np.sqrt(np.diag(cov_res))
    fit_ok   = True
    print(f'Resolution: sigma = {a_res:.3f}+-{perr_res[0]:.3f} + ({b_res:.3f}+-{perr_res[1]:.3f})*E_true')
except Exception:
    fit_ok = False

# =============================================================================
# PLOT 1: per-slice diagnostic
# =============================================================================
ncols = 4
nrows = int(np.ceil(N_BINS / ncols))
fig_s, axes_s = plt.subplots(nrows, ncols, figsize=(15, 3.5*nrows),
                              gridspec_kw={'hspace':0.60,'wspace':0.38})
axes_s = axes_s.flatten()

for i, (lo, hi) in enumerate(zip(E_TRUE_BINS[:-1], E_TRUE_BINS[1:])):
    ax    = axes_s[i]
    er_b  = slice_data[i]
    label = fr'$E_{{true}}\in[{lo:.2f},{hi:.2f}]$ GeV'

    if slice_counts[i] is not None and len(er_b) > 8:
        counts = slice_counts[i]
        edges  = slice_edges[i]
        cens   = 0.5*(edges[:-1]+edges[1:])
        width  = edges[1]-edges[0]
        errs   = np.sqrt(np.maximum(counts, 1))

        ax.bar(cens, counts, width=width*0.85,
               color=BLUE, alpha=0.55, zorder=2)
        ax.errorbar(cens, counts, yerr=errs, fmt='none',
                    color=BLUE, lw=1.2, capsize=2, zorder=3)

        if slice_popt[i] is not None:
            x_fit = np.linspace(edges[0], edges[-1], 300)
            y_fit = bifurcated_gaussian(x_fit, *slice_popt[i])
            ax.plot(x_fit, y_fit, color=CRIMSON, lw=2, zorder=5,
                    label='Bifurc. fit')

        if ok_p[i]:
            mpv = peak_reco[i]
            ax.axvline(mpv, color=CRIMSON, lw=2, ls='--', zorder=6,
                       label=f'MPV={mpv:.2f} GeV')
            ax.axvspan(mpv-peak_err[i], mpv+peak_err[i],
                       color=CRIMSON, alpha=0.12, zorder=4)
            ridge_pred = m*E_TRUE_CEN[i] + c
            ax.axvline(ridge_pred, color=GOLD, lw=1.5, ls=':', zorder=6,
                       label=f'Ridge={ridge_pred:.2f} GeV')
            ax.legend(fontsize=6.5, loc='upper right')

        # enforce x-axis to physical window
        expected = 0.5*(lo+hi)
        ax.set_xlim(max(0, expected-0.75), min(3.5, expected+0.75))
        ax.text(0.04, 0.96, f'N={len(er_b)}', transform=ax.transAxes,
                fontsize=7, va='top', color='#444444')
    else:
        ax.text(0.5, 0.5, f'{label}\nInsufficient stats\n(N={len(er_b)})',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=8, color='#888888')
        expected = 0.5*(lo+hi)
        ax.set_xlim(max(0, expected-0.75), min(3.5, expected+0.75))

    ax.set_title(label, fontsize=8)
    ax.set_xlabel(r'Reco $E_{\nu}$ [GeV]', fontsize=8)
    ax.set_ylabel('Events', fontsize=8)
    ax.tick_params(labelsize=7)

for j in range(i+1, len(axes_s)):
    axes_s[j].set_visible(False)

fig_s.suptitle(
    r'Reco $E_{\nu}$ per true $E_{\nu}$ slice'
    '\n(bifurcated Gaussian fit | crimson = MPV | gold = ridge prediction)',
    fontsize=11)
#savefig(fig_s, 'energy_slices.png')
plt.show()

# =============================================================================
# PLOT 2: main resolution figure
# =============================================================================
fig = plt.figure(figsize=(14,10))
gs  = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.35)
ax_main = fig.add_subplot(gs[0,:])
ax_res  = fig.add_subplot(gs[1,0])
ax_sg   = fig.add_subplot(gs[1,1])

# main panel
h,xb,yb,im = ax_main.hist2d(Et, Er, bins=32, range=[[0,3],[0,3]], cmap=MANAGUA)
plt.colorbar(im, ax=ax_main, label='Events')
ax_main.plot([0,3],[0,3], 'w--', lw=1.8, label='Perfect reconstruction', zorder=5)
ax_main.errorbar(E_TRUE_CEN[ok_p], peak_reco[ok_p], yerr=peak_err[ok_p],
                 fmt='o', ms=5.5, color='white', markeredgecolor=CRIMSON,
                 markeredgewidth=1.8, capsize=3, lw=1.4, zorder=7,
                 label='Bifurc. Gaussian MPV per slice')
ax_main.plot(Ef, Ep, color=CRIMSON, lw=2.5, zorder=6,
             label=fr'Weighted MPV fit: $E_{{reco}}={m:.3f}\pm{m_err:.3f}'
                   fr'\,E_{{true}}{c:+.3f}\pm{c_err:.3f}$  ($R^2={r_val_sq:.3f}$)')
ax_main.fill_between(Ef, Ep-band, Ep+band, color=CRIMSON, alpha=0.20,
                     zorder=4, label=r'$1\sigma$ fit uncertainty')
ax_main.axvline(FIT_LO, ls=':', lw=1, color=CRIMSON, alpha=0.4)
ax_main.axvline(FIT_HI, ls=':', lw=1, color=CRIMSON, alpha=0.4)
ax_main.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_main.set_ylabel(r'Reco $E_{\nu}$ [GeV]')
ax_main.set_title(r'Neutrino energy: reco vs true (selected signal, no trigger, Pandora)')
ax_main.legend(fontsize=8.5, loc='upper left')
ax_main.set_xlim(0,3); ax_main.set_ylim(0,3)

# residual panel
ax_res.axhline(0, ls='--', lw=1.5, color='#555555', zorder=5)
ax_res.fill_between(E_TRUE_CEN[ok_b], (r_mu-r_sg)[ok_b], (r_mu+r_sg)[ok_b],
                    color=BLUE, alpha=0.15, step='mid', zorder=2,
                    label=r'$\pm 1\sigma$ spread around ridge')
ax_res.errorbar(E_TRUE_CEN[ok_p], r_mu[ok_p], yerr=r_err[ok_p],
                fmt='o', ms=6, lw=1.8, capsize=4, color=BLUE,
                markeredgecolor='white', markeredgewidth=0.5,
                label='MPV residual', zorder=5)
ax_res.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_res.set_ylabel(r'$(E_{MPV}-E_{true})/E_{true}$')
ax_res.set_title('MPV fractional residual vs energy')
ax_res.set_ylim(-0.5, 0.3)
ax_res.legend(fontsize=9)

# resolution panel
Ef_res = np.linspace(FIT_LO, FIT_HI, 200)
if fit_ok:
    ax_sg.plot(Ef_res, a_res+b_res*Ef_res, color=CRIMSON, lw=2.2, zorder=4,
               label=fr'Fit: $\sigma={a_res:.3f}\pm{perr_res[0]:.3f}'
                     fr'+({b_res:.3f}\pm{perr_res[1]:.3f})\,E_{{true}}$')
    ax_sg.fill_between(Ef_res,
        np.maximum(0,(a_res-perr_res[0])+(b_res-perr_res[1])*Ef_res),
        (a_res+perr_res[0])+(b_res+perr_res[1])*Ef_res,
        color=CRIMSON, alpha=0.18, zorder=3)

ax_sg.fill_between(E_TRUE_CEN[ok_b],
                   np.maximum(0,(r_sg-r_err)[ok_b]), (r_sg+r_err)[ok_b],
                   color=BLUE, alpha=0.18, step='mid', zorder=2)
ax_sg.plot(E_TRUE_CEN[ok_b], r_sg[ok_b], 'o', color=BLUE, ms=6, zorder=5,
           markeredgecolor='white', markeredgewidth=0.5,
           label='Resolution (RMS around ridge)')
ax_sg.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_sg.set_ylabel(r'$\sigma[(E_{reco}-E_{ridge})/E_{true}]$')
ax_sg.set_title('Energy resolution (RMS around ridge) vs energy')
ax_sg.set_ylim(0, 0.45)
ax_sg.legend(fontsize=8)

#savefig(fig, 'energy_resolution.png')
plt.show()

### Why residuals should be symmetric around zero
If the bifurcated Gaussian is correctly finding the physical peak of the reco energy distribution in each E_true bin, then by construction the residual (MPV - E_true_bin_centre)/E_true_bin_centre measures the
systematic energy scale offset. This should be a smooth, slowly varying
function of E_true (reflecting physical effects like missing neutron
energy) and the points should scatter symmetrically around this curve.
Any remaining asymmetry indicates either a fit convergence issue in a
specific bin (visible in the slice diagnostic plots) or genuine physical non-linearity in the energy response. In a few bins you also notice there aren't a lot of statistics - this is also an important consideration to note.

### Current status
The bifurcated Gaussian gives the most reliable MPV estimates and the
most symmetric residuals of the methods tried. The fit occasionally fails in low-statistics bins (N < 15) where the optimiser cannot constrain all four parameters simultaneously; these bins fall back to a KDE estimate. The energy scale (fit slope ~0.91) and resolution (~14-17%) are consistent with expectations for calorimetric reconstruction in LArTPC with missing hadronic energy.

### Selection purity vs energy resolution

The plot below shows selection purity as a function of the neutrino energy resolution $(E_{\nu}^{\mathrm{reco}} - E_{\nu}^{\mathrm{true}}) / E_{\nu}^{\mathrm{true}}$. This is the direct analogue of the purity-vs-energy plot but in resolution space, and reveals whether individual cuts preferentially remove signal events that were poorly reconstructed (e.g., large negative resolution, consistent with missing hadronic energy).

In [ ]:
# Selection purity vs neutrino energy resolution (E_reco - E_true) / E_true
# This shows where in resolution-space each cut stage gains or loses purity.
# The backdrop histogram is the signal event density in resolution bins.

RES_BINS = np.linspace(-1.0, 0.6, 33)   # (E_reco - E_true) / E_true
RES_CEN  = 0.5*(RES_BINS[:-1] + RES_BINS[1:])
RES_WID  = np.diff(RES_BINS)

ok_res = (
    df['reco_nu_energy'].notna() & np.isfinite(df['reco_nu_energy']) & (df['reco_nu_energy'] > 0) &
    df['true_nu_energy'].notna() & np.isfinite(df['true_nu_energy']) & (df['true_nu_energy'] > 0)
)

res_all = ((df['reco_nu_energy'] - df['true_nu_energy']) / df['true_nu_energy']).where(ok_res)

fig, ax = plt.subplots(figsize=(10, 5.5), dpi=300)

for (label, cut_fn), col in zip(CUT_STEPS, SEL_COLS):
    mask = cut_fn(df) & ok_res
    mask_sig = mask & sig

    res_sig  = res_all[mask_sig].dropna()
    res_sel  = res_all[mask].dropna()

    n_sig_b, _ = np.histogram(res_sig, bins=RES_BINS)
    n_sel_b, _ = np.histogram(res_sel, bins=RES_BINS)

    pur = np.where(n_sel_b > 0, n_sig_b / n_sel_b, np.nan)
    err = np.where(n_sel_b > 0, np.sqrt(pur*(1-pur)/np.where(n_sel_b>0, n_sel_b, 1)), np.nan)
    tot_pur = (cut_fn(df) & sig).sum() / cut_fn(df).sum() if cut_fn(df).sum() > 0 else np.nan

    ax.fill_between(RES_CEN, pur-err, pur+err,
                    color=col, alpha=0.15, step='mid', zorder=2)
    ax.errorbar(RES_CEN, pur, xerr=RES_WID/2,
                fmt='o', ms=4.5, lw=1.8, capsize=0, color=col,
                markeredgecolor='white', markeredgewidth=0.5,
                label=f'{label} ({100*tot_pur:.0f}%)', zorder=5)

# Signal event density backdrop
ax2 = ax.twinx()
ax2.hist(res_all[sig & ok_res].dropna(), bins=RES_BINS,
         color='#bdbdbd', alpha=0.22, zorder=0)
ax2.set_ylabel('Signal events', color='#999999', fontsize=11)
ax2.tick_params(axis='y', colors='#999999')
ax2.set_ylim(bottom=0)
ax2.grid(False)

ax.axhline(1.0, ls='--', lw=1.5, color='#555555', zorder=5)
ax.axvline(0.0, ls=':', lw=1.2, color='#888888', zorder=4)
ax.set_xlabel(r'$(E_{\nu}^{\mathrm{reco}} - E_{\nu}^{\mathrm{true}}) \,/\, E_{\nu}^{\mathrm{true}}$')
ax.set_ylabel('Selection purity')
ax.set_xlim(-1.0, 0.6)
ax.set_ylim(0, 1.35)
ax.legend(ncol=3, loc='upper left', fontsize=9)
ax.set_title('CC1e0pi selection purity vs energy resolution (Pandora-only)')
fig.tight_layout()
plt.show()

---
# Cut Optimisation Studies

The nominal selection was tuned on NuMI, where the neutrino flux peaks around 2 GeV.
BNB has a harder spectrum peaking closer to 0.7 GeV, so some cuts -- especially the
shower-energy floor and the shower-track score threshold -- may not be at their optimal
working point.

This section asks: **can we improve purity without meaningfully sacrificing efficiency?**
The strategy is:

1. Verify the dE/dx cut behaviour on BNB showers.
2. Map out 1D efficiency-purity frontiers for every cut variable.
3. Scan correlated 2D cut planes (energy vs dE/dx, angle vs gap, energy vs shower score).
4. Define optimised working points and compare them against the nominal selection.

In [ ]:
# Metric helpers used throughout this section.

def selection_metrics(sel, sig, label=''):
    n_true_sig = sig.sum()
    n_sel      = sel.sum()
    n_sig      = (sel & sig).sum()
    n_bkg      = n_sel - n_sig
    eff  = n_sig / n_true_sig if n_true_sig > 0 else np.nan
    pur  = n_sig / n_sel      if n_sel      > 0 else np.nan
    fom  = eff / (1 + np.sqrt(max(n_bkg, 0)))        # Punzi FOM
    ssb  = n_sig / np.sqrt(n_sel) if n_sel > 0 else np.nan
    return {'Selection': label, 'N sel': n_sel, 'N sig': n_sig, 'N bkg': n_bkg,
            'Eff. (%)': 100*eff, 'Pur. (%)': 100*pur,
            'Punzi FOM': fom, r'$S/\sqrt{S+B}$': ssb}

def make_metrics_table(configs, sig):
    rows = [selection_metrics(sel, sig, name) for name, sel in configs.items()]
    return pd.DataFrame(rows).round({'Eff. (%)': 2, 'Pur. (%)': 2,
                                     'Punzi FOM': 5, r'$S/\sqrt{S+B}$': 2})

def scan_1d(vals, sel_fn, sig):
    rows = []
    for v in vals:
        m = selection_metrics(sel_fn(v), sig, label=f'{v:.3g}')
        m['Cut value'] = v
        rows.append(m)
    return pd.DataFrame(rows)

base_metrics = selection_metrics(sel_full(df), sig, 'Nominal')
base_eff = base_metrics['Eff. (%)']
base_pur = base_metrics['Pur. (%)']
print(f'Nominal  ->  Eff. = {base_eff:.1f}%   Pur. = {base_pur:.1f}%')

## Step 1 -- dE/dx behaviour on BNB showers

The dE/dx cut (<3.5 MeV/cm) separates electromagnetic showers (dE/dx ~2 MeV/cm)
from hadronic activity and photon conversions. On NuMI this cut costs about 8% in
efficiency but significantly suppresses $\pi^0$ backgrounds.

Below we check (i) the raw dE/dx distribution for signal and background after the
full selection is applied up to -- but not including -- the dE/dx cut, and (ii)
the efficiency-purity trade-off at three alternative thresholds.

In [ ]:
DEDX_COL = 'shower_avail_dedx'

# Mask: pass everything up to dE/dx
pre_dedx = sel_shower(df)  # shower energy already applied

# ---- Panel 1: dE/dx distribution ----
dedx_sig = df.loc[pre_dedx & sig,  DEDX_COL].dropna()
dedx_bkg = df.loc[pre_dedx & ~sig, DEDX_COL].dropna()
bins_dx  = np.linspace(0, 10, 50)

# ---- Panel 2: eff/purity vs threshold ----
configs_dedx = {
    'No cut':         pre_dedx & c_angle(df) & c_gap(df) & c_proton(df) & c_0pi(df) & c_muon(df),
    r'<2.5 MeV/cm':  pre_dedx & c_dedx(df,2.5) & c_angle(df) & c_gap(df) & c_proton(df) & c_0pi(df) & c_muon(df),
    r'<3.5 (nominal)':pre_dedx & c_dedx(df,3.5) & c_angle(df) & c_gap(df) & c_proton(df) & c_0pi(df) & c_muon(df),
    r'<5.0 MeV/cm':  pre_dedx & c_dedx(df,5.0) & c_angle(df) & c_gap(df) & c_proton(df) & c_0pi(df) & c_muon(df),
}
dx_table = make_metrics_table(configs_dedx, sig)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.0), dpi=300,
                                gridspec_kw={'wspace': 0.38})

# -- distribution --
ax1.hist(dedx_sig, bins=bins_dx, histtype='stepfilled',
         color=BLUE,   alpha=0.35, density=True, label='Signal')
ax1.hist(dedx_bkg, bins=bins_dx, histtype='stepfilled',
         color=CRIMSON,alpha=0.35, density=True, label='Background')
ax1.hist(dedx_sig, bins=bins_dx, histtype='step', lw=1.8,
         color=BLUE,   density=True)
ax1.hist(dedx_bkg, bins=bins_dx, histtype='step', lw=1.8,
         color=CRIMSON,density=True)
ax1.axvline(SHOWER_DEDX_MAX, ls='--', lw=1.8, color='#555555',
            label=f'Nominal cut ({SHOWER_DEDX_MAX} MeV/cm)')
ax1.set_xlabel(r'Shower dE/dx [MeV/cm]')
ax1.set_ylabel('Area-normalised entries')
ax1.set_title('dE/dx distribution after shower-energy cut')
ax1.set_xlim(0, 10)
ax1.legend(fontsize=9)

# -- efficiency / purity bars --
x    = np.arange(len(dx_table))
COLS_BARS = [BLUE, CRIMSON, GOLD, '#3f8f3f']
for xi, (_, row), col in zip(x, dx_table.iterrows(), COLS_BARS):
    ax2.bar(xi - 0.20, row['Eff. (%)'], 0.38, color=col,   alpha=0.85, zorder=3)
    ax2.bar(xi + 0.20, row['Pur. (%)'], 0.38, color=col,   alpha=0.45, zorder=3,
            edgecolor=col, linewidth=1.5, fill=True, hatch='//')
    ax2.text(xi-0.20, row['Eff. (%)']+0.8, f"{row['Eff. (%)']:.1f}",
             ha='center', va='bottom', fontsize=8, color=col, fontweight='bold')
    ax2.text(xi+0.20, row['Pur. (%)']+0.8, f"{row['Pur. (%)']:.1f}",
             ha='center', va='bottom', fontsize=8, color=col, fontweight='bold')

from matplotlib.patches import Patch
leg_els = [Patch(facecolor='#888888', alpha=0.85, label='Efficiency (solid)'),
           Patch(facecolor='#888888', alpha=0.45, hatch='//', label='Purity (hatched)')]
ax2.legend(handles=leg_els, fontsize=9)
ax2.set_xticks(x)
ax2.set_xticklabels(dx_table['Selection'], rotation=20, ha='right', fontsize=10)
ax2.set_ylabel('Percentage (%)')
ax2.set_title('Efficiency and purity at each dE/dx threshold')
ax2.set_ylim(0, max(dx_table[['Eff. (%)','Pur. (%)']].max()) * 1.18)

fig.suptitle(r'dE/dx cut behaviour on BNB (Pandora-only)', fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

## Step 2 -- 1D efficiency-purity frontiers

Each cut is swept independently while all others are held at their nominal values.
The top row shows efficiency and purity as a function of the cut value; the bottom
panel overlays all five frontiers on a single efficiency-purity plane.
The **crimson star** marks the nominal working point; the **gold diamond** marks the
point that maximises purity subject to retaining >=90% of the nominal efficiency.

In [ ]:
scan_defs = [
    {'label': 'Shower $E$ min',       'xlabel': r'Shower energy min [GeV]',  'vals': np.linspace(0.05, 0.70, 70), 'base': SHOWER_ENERGY_MIN,  'fn': lambda v: sel_flash(df)  & c_shower(df,v) & c_dedx(df)   & c_angle(df) & c_gap(df) & c_proton(df) & c_0pi(df) & c_muon(df)},
    {'label': 'dE/dx max',            'xlabel': r'Shower dE/dx max [MeV/cm]','vals': np.linspace(1.0,  8.0,  70), 'base': SHOWER_DEDX_MAX,    'fn': lambda v: sel_shower(df) & c_dedx(df,v)   & c_angle(df) & c_gap(df) & c_proton(df) & c_0pi(df) & c_muon(df)},
    {'label': 'Open angle max',       'xlabel': r'Open angle max [deg]',     'vals': np.linspace(2.0, 40.0,  70), 'base': SHOWER_ANGLE_MAX,   'fn': lambda v: sel_dedx(df)   & c_angle(df,v)  & c_gap(df)   & c_proton(df) & c_0pi(df) & c_muon(df)},
    {'label': 'Conv. gap max',        'xlabel': r'Conv. gap max [cm]',       'vals': np.linspace(0.5, 15.0,  70), 'base': SHOWER_CONVGAP_MAX, 'fn': lambda v: sel_dedx(df)   & c_angle(df)    & c_gap(df,v) & c_proton(df) & c_0pi(df) & c_muon(df)},
    {'label': r'$\chi^2_\mu$ veto',   'xlabel': r'$\chi^2_\mu$ veto max',   'vals': np.linspace(5.0,100.0,  70), 'base': MUON_CHI2_MU_MAX,  'fn': lambda v: sel_0pi(df)    & c_muon(df,v)},
]
SCAN_COLS = SEL_COLS[2:]  # skip first two (too dark for lines)

scan_results = {}
best_pts     = {}

for s in scan_defs:
    out = scan_1d(s['vals'], s['fn'], sig)
    out['Scan'] = s['label']
    scan_results[s['label']] = out
    # best purity at >=90% efficiency
    allowed = out[out['Eff. (%)'] >= 0.90*base_eff]
    if len(allowed):
        row = allowed.sort_values('Pur. (%)', ascending=False).iloc[0]
        best_pts[s['label']] = {'v': row['Cut value'], 'eff': row['Eff. (%)'], 'pur': row['Pur. (%)']}

# ---- Figure: per-cut panels (2 rows: efficiency, purity) + Pareto map ------
n_cuts = len(scan_defs)
fig = plt.figure(figsize=(4.2*n_cuts, 11), dpi=300)
gs  = fig.add_gridspec(3, n_cuts, hspace=0.52, wspace=0.38)

for col, (s, col_c) in enumerate(zip(scan_defs, SCAN_COLS)):
    out  = scan_results[s['label']]
    bpt  = best_pts.get(s['label'], None)

    for row, (ykey, ylabel) in enumerate([('Eff. (%)', 'Efficiency (%)'), ('Pur. (%)', 'Purity (%)')]):
        ax = fig.add_subplot(gs[row, col])
        ax.plot(out['Cut value'], out[ykey], '-', color=col_c, lw=2, zorder=4)
        ax.axvline(s['base'],  ls='--', lw=1.6, color=CRIMSON, alpha=0.85, label='Nominal')
        if bpt:
            ax.axvline(bpt['v'], ls=':',  lw=1.6, color=GOLD,   alpha=0.90, label='90% eff. opt.')
        ax.set_xlabel(s['xlabel'], fontsize=9)
        ax.set_ylabel(ylabel, fontsize=10)
        if row == 0:
            ax.set_title(s['label'], fontsize=10)
            ax.legend(fontsize=7, loc='best')
        ax.set_ylim(bottom=0)

# ---- Pareto map (bottom row, full width) -----------------------------------
ax_p = fig.add_subplot(gs[2, :])

for s, col_c in zip(scan_defs, SCAN_COLS):
    out = scan_results[s['label']]
    ax_p.plot(out['Eff. (%)'], out['Pur. (%)'], 'o-',
              ms=3.5, lw=1.7, color=col_c,
              markeredgecolor='white', markeredgewidth=0.35,
              label=s['label'], alpha=0.95)
    bpt = best_pts.get(s['label'])
    if bpt:
        ax_p.scatter(bpt['eff'], bpt['pur'],
                     marker='D', s=70, color=col_c,
                     edgecolor='white', linewidth=0.7, zorder=7)

ax_p.scatter(base_eff, base_pur,
             marker='*', s=260, color=CRIMSON,
             edgecolor='white', linewidth=0.9,
             zorder=9, label='Nominal')
ax_p.set_xlabel('Efficiency (%)')
ax_p.set_ylabel('Purity (%)')
ax_p.set_title('Efficiency-purity Pareto frontier (diamonds = 90% eff. optimum per cut)')
ax_p.legend(ncol=3, fontsize=8, loc='best')
ax_p.set_xlim(left=0); ax_p.set_ylim(bottom=0)

fig.suptitle('1D cut sweeps -- BNB Pandora-only', fontsize=13, y=1.01)
plt.show()

## Step 3 -- 2D correlated cut scans

Two-dimensional scans reveal how pairs of cuts trade off against each other.
Because the angle and gap cuts both reject extended activity near the vertex,
their optimal pair differs from two independent 1D optima.
Similarly, shower energy and shower-track score are anti-correlated: events
rejected by a tighter score threshold are often soft, so raising the energy floor
in tandem partially recovers efficiency.

Each map shows **purity** as the colour scale with **efficiency contours** overlaid.
The crimson star is the nominal working point; the gold diamond is the maximum-purity
point that retains >=90% of nominal efficiency.

In [ ]:
# ---- Shared helper: run a 2D scan and find the best-purity point -------------
def run_2d_scan(x_vals, y_vals, sel_fn):
    eff_m = np.full((len(y_vals), len(x_vals)), np.nan)
    pur_m = np.full_like(eff_m, np.nan)
    for i, yv in enumerate(y_vals):
        for j, xv in enumerate(x_vals):
            m = selection_metrics(sel_fn(xv, yv), sig)
            eff_m[i, j] = m['Eff. (%)']
            pur_m[i, j] = m['Pur. (%)']
    allowed = np.where(eff_m >= 0.90*base_eff, pur_m, np.nan)
    if np.any(np.isfinite(allowed)):
        ib, jb = np.unravel_index(np.nanargmax(allowed), allowed.shape)
    else:
        ib = jb = None
    return eff_m, pur_m, ib, jb

def draw_2d_map(ax, x_vals, y_vals, eff_m, pur_m, ib, jb,
                xlabel, ylabel, nom_x, nom_y, title):
    vmax = max(50, np.nanpercentile(pur_m, 98))
    im = ax.pcolormesh(x_vals, y_vals, pur_m,
                       shading='auto', cmap=MANAGUA, vmin=0, vmax=vmax)
    plt.colorbar(im, ax=ax, label='Purity (%)')
    levs = [l for l in [10,15,20,25,30,35,40,45]
            if np.nanmin(eff_m) <= l <= np.nanmax(eff_m)]
    if levs:
        cs = ax.contour(x_vals, y_vals, eff_m, levels=levs,
                        colors='white', linewidths=0.8, alpha=0.7)
        ax.clabel(cs, inline=True, fontsize=7, fmt='%.0f%%')
    ax.scatter([nom_x], [nom_y], marker='*', s=200,
               color=CRIMSON, edgecolor='white', lw=0.8, zorder=8, label='Nominal')
    if ib is not None:
        ax.scatter([x_vals[jb]], [y_vals[ib]], marker='D', s=80,
                   color=GOLD, edgecolor='white', lw=0.7, zorder=8,
                   label=r'Best pur., $\geq$90% eff.')
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(fontsize=8, loc='best')

# ---- Scan 1: shower energy vs dE/dx ----------------------------------------
E_SC   = np.linspace(0.05, 0.60, 45)
DX_SC  = np.linspace(1.0,  8.0,  45)
fn_edx = lambda xv, yv: (sel_flash(df) & c_shower(df,yv) & c_dedx(df,xv)
                          & c_angle(df) & c_gap(df) & c_proton(df) & c_0pi(df) & c_muon(df))
eff_edx, pur_edx, ib_edx, jb_edx = run_2d_scan(DX_SC, E_SC, fn_edx)

# ---- Scan 2: open angle vs conversion gap -----------------------------------
ANG_SC = np.linspace(2.0, 40.0, 45)
GAP_SC = np.linspace(0.5, 15.0, 45)
fn_ag  = lambda xv, yv: (sel_dedx(df) & c_angle(df,yv) & c_gap(df,xv)
                          & c_proton(df) & c_0pi(df) & c_muon(df))
eff_ag, pur_ag, ib_ag, jb_ag = run_2d_scan(GAP_SC, ANG_SC, fn_ag)

# ---- Scan 3: shower energy vs shower-track score ----------------------------
SCORE_COL   = 'shower_track_score'
valid_score = df[SCORE_COL].notna() & (df[SCORE_COL] > 0)
SC_SC       = np.linspace(np.nanpercentile(df.loc[valid_score,SCORE_COL],1),
                           np.nanpercentile(df.loc[valid_score,SCORE_COL],99), 45)
fn_es  = lambda xv, yv: (valid_score & sel_flash(df) & c_shower(df,yv)
                          & (df[SCORE_COL] < xv) & c_dedx(df) & c_angle(df)
                          & c_gap(df) & c_proton(df) & c_0pi(df) & c_muon(df))
eff_es, pur_es, ib_es, jb_es = run_2d_scan(SC_SC, E_SC, fn_es)

# ---- Plot all three side by side -------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(19, 6.0), dpi=300,
                          gridspec_kw={'wspace': 0.42})

draw_2d_map(axes[0], DX_SC,  E_SC,   eff_edx, pur_edx, ib_edx, jb_edx,
            r'dE/dx max [MeV/cm]', r'Shower $E$ min [GeV]',
            SHOWER_DEDX_MAX, SHOWER_ENERGY_MIN,
            r'Shower energy vs dE/dx')

draw_2d_map(axes[1], GAP_SC, ANG_SC, eff_ag,  pur_ag,  ib_ag,  jb_ag,
            r'Conv. gap max [cm]', r'Open angle max [deg]',
            SHOWER_CONVGAP_MAX, SHOWER_ANGLE_MAX,
            r'Angle vs conversion gap')

draw_2d_map(axes[2], SC_SC,  E_SC,   eff_es,  pur_es,  ib_es,  jb_es,
            r'Track score max (shower $\leftarrow$ 0, track $\rightarrow$ 1)',
            r'Shower $E$ min [GeV]',
            np.nanmax(SC_SC), SHOWER_ENERGY_MIN,
            r'Shower energy vs track score')

fig.suptitle('2D purity maps with efficiency contours (BNB Pandora-only)', fontsize=13, y=1.01)
plt.show()

# Report best points
for label, sc_x, sc_y, ib, jb, xvals, yvals in [
    ('E_sh vs dE/dx',  DX_SC,  E_SC,   ib_edx, jb_edx, DX_SC, E_SC),
    ('Angle vs gap',   GAP_SC, ANG_SC, ib_ag,  jb_ag,  GAP_SC, ANG_SC),
    ('E_sh vs score',  SC_SC,  E_SC,   ib_es,  jb_es,  SC_SC, E_SC),
]:
    if ib is not None:
        print(f'{label}: best pur. point at x={xvals[jb]:.3f}, y={yvals[ib]:.3f}')

### Shower-track score: signal vs background separation

Before committing to a tighter score cut, it is worth checking that the signal and
background populations actually separate in `shower_track_score`.
The plot below shows area-normalised distributions for signal and background
events passing the full nominal selection.

In [ ]:
SCORE_COL = 'shower_track_score'

if SCORE_COL in df.columns:
    sel_pre = sel_full(df) & df[SCORE_COL].notna()

    bins_sc = np.linspace(
        np.nanpercentile(df.loc[sel_pre, SCORE_COL], 0.5),
        np.nanpercentile(df.loc[sel_pre, SCORE_COL], 99.5),
        45
    )

    fig, ax = plt.subplots(figsize=(8.5, 5.0), dpi=300)

    ax.hist(df.loc[sel_pre & sig,  SCORE_COL].dropna(),
            bins=bins_sc, histtype='stepfilled', density=True,
            color=BLUE,    alpha=0.35, label='Signal')
    ax.hist(df.loc[sel_pre & ~sig, SCORE_COL].dropna(),
            bins=bins_sc, histtype='stepfilled', density=True,
            color=CRIMSON, alpha=0.35, label='Background')
    ax.hist(df.loc[sel_pre & sig,  SCORE_COL].dropna(),
            bins=bins_sc, histtype='step', lw=2.0, density=True, color=BLUE)
    ax.hist(df.loc[sel_pre & ~sig, SCORE_COL].dropna(),
            bins=bins_sc, histtype='step', lw=2.0, density=True, color=CRIMSON)

    ax.axvline(0.5, ls='--', lw=1.8, color='#555555', label='Nominal cut (score < 0.5)')
    ax.set_xlabel(SCORE_COL)
    ax.set_ylabel('Area-normalised entries')
    ax.set_title('Signal/background separation in shower-track score (post-selection)')
    ax.legend(fontsize=10)
    fig.tight_layout()
    plt.show()
else:
    print(f'{SCORE_COL} not found in ntuple.')

## Step 4 -- Optimised working points

Based on the 1D and 2D scans, we define two alternative working points:

- **Retuned shower energy** -- raise the shower-energy floor to the value that
  maximises purity while retaining >=90% of the nominal efficiency (read from
  the 1D scan above; default placeholder shown, update after running).
- **Retuned shower energy + track score** -- additionally tighten the shower-track
  score threshold using the 2D scan optimum.

The cells below define these selections, print a global metrics table,
and compare them against the nominal in binned $E_\nu^\mathrm{true}$ plots.

In [ ]:
# Working points from the scans above.
# Update these after inspecting the printed output from Step 2-3.
RETUNED_SHOWER_ENERGY_MIN = 0.314   # from 1D Pareto scan (90% eff. retention)

SCORE_COL   = 'shower_track_score'
valid_score = df[SCORE_COL].notna() & (df[SCORE_COL] > 0)

# From the 2D E_sh vs score scan (replace with best_pts values if still in memory)
RETUNED_COMBINED_SHOWER_ENERGY_MIN = 0.273   # update after running scan
RETUNED_SCORE_MAX                  = 0.479   # update after running scan

configs_retuned = {
    'Nominal': sel_full(df),

    rf'Retuned $E_{{sh}}>$ {RETUNED_SHOWER_ENERGY_MIN:.3f} GeV': (
        sel_flash(df)
        & c_shower(df, RETUNED_SHOWER_ENERGY_MIN)
        & c_dedx(df) & c_angle(df) & c_gap(df)
        & c_proton(df) & c_0pi(df) & c_muon(df)
    ),

    (rf'Retuned $E_{{sh}}>$ {RETUNED_COMBINED_SHOWER_ENERGY_MIN:.3f} GeV, '
     rf'score $<$ {RETUNED_SCORE_MAX:.3f}'): (
        valid_score
        & sel_flash(df)
        & c_shower(df, RETUNED_COMBINED_SHOWER_ENERGY_MIN)
        & (df[SCORE_COL] < RETUNED_SCORE_MAX)
        & c_dedx(df) & c_angle(df) & c_gap(df)
        & c_proton(df) & c_0pi(df) & c_muon(df)
    ),
}

# Global metrics table
print('Global selection metrics:')
make_metrics_table(configs_retuned, sig)

In [ ]:
# Efficiency and purity vs true neutrino energy for nominal vs retuned selections.
# Plotted side by side for direct comparison.

E_BINS_C = np.array([0.0,0.2,0.4,0.6,0.8,1.0,1.2,1.4,1.6,1.8,2.0,2.4,2.8,3.2,4.0])
E_CEN_C  = 0.5*(E_BINS_C[:-1]+E_BINS_C[1:])
E_WID_C  = np.diff(E_BINS_C)

CMP_COLS = [BLUE, CRIMSON, GOLD]
CUT_STEPS_COMPARE = list(configs_retuned.items())

fig, (ax_e, ax_p) = plt.subplots(1, 2, figsize=(16, 5.5), dpi=300,
                                   gridspec_kw={'wspace': 0.32})

denom, _ = np.histogram(df.loc[sig, 'true_nu_energy'].dropna(), bins=E_BINS_C)

for (label, mask), col in zip(CUT_STEPS_COMPARE, CMP_COLS):
    # efficiency
    num, _  = np.histogram(df.loc[mask & sig, 'true_nu_energy'].dropna(), bins=E_BINS_C)
    eff     = np.where(denom > 0, num/denom, np.nan)
    eff_err = np.where(denom > 0, np.sqrt(eff*(1-eff)/np.where(denom>0,denom,1)), np.nan)
    tot_eff = (mask & sig).sum() / sig.sum()

    # purity
    n_sig_b, _ = np.histogram(df.loc[mask & sig, 'true_nu_energy'].dropna(), bins=E_BINS_C)
    n_sel_b, _ = np.histogram(df.loc[mask,        'true_nu_energy'].dropna(), bins=E_BINS_C)
    pur     = np.where(n_sel_b > 0, n_sig_b/n_sel_b, np.nan)
    pur_err = np.where(n_sel_b > 0, np.sqrt(pur*(1-pur)/np.where(n_sel_b>0,n_sel_b,1)), np.nan)
    tot_pur = (mask & sig).sum() / mask.sum() if mask.sum() > 0 else np.nan

    kw = dict(fmt='o', ms=4.5, lw=1.8, capsize=0, color=col,
              markeredgecolor='white', markeredgewidth=0.5, zorder=5)

    ax_e.fill_between(E_CEN_C, eff-eff_err, eff+eff_err, color=col, alpha=0.14, step='mid', zorder=2)
    ax_e.errorbar(E_CEN_C, eff, xerr=E_WID_C/2, label=f'{label} ({100*tot_eff:.1f}%)', **kw)

    ax_p.fill_between(E_CEN_C, pur-pur_err, pur+pur_err, color=col, alpha=0.14, step='mid', zorder=2)
    ax_p.errorbar(E_CEN_C, pur, xerr=E_WID_C/2, label=f'{label} ({100*tot_pur:.1f}%)', **kw)

for ax, col2_lab, ylabel, ylim in [
    (ax_e, 'Signal events',           'Selection efficiency',  (0, 1.35)),
    (ax_p, 'Selected events (nominal)', 'Selection purity',   (0, 1.35)),
]:
    ax2 = ax.twinx()
    ax2.hist(df.loc[sig, 'true_nu_energy'].dropna(), bins=E_BINS_C,
             color='#bdbdbd', alpha=0.20, zorder=0)
    ax2.set_ylabel(col2_lab, color='#999999', fontsize=11)
    ax2.tick_params(axis='y', colors='#999999'); ax2.set_ylim(bottom=0); ax2.grid(False)
    ax.axhline(1.0, ls='--', lw=1.4, color='#555555', zorder=5)
    ax.set_xlabel(r'$E_{\nu}^{\mathrm{true}}$ [GeV]')
    ax.set_ylabel(ylabel)
    ax.set_xlim(0, 4); ax.set_ylim(*ylim)
    ax.legend(ncol=1, loc='upper right', fontsize=9)

ax_e.set_title(r'Efficiency vs $E_{\nu}^{\mathrm{true}}$')
ax_p.set_title(r'Purity vs $E_{\nu}^{\mathrm{true}}$')
fig.suptitle('Nominal vs retuned: binned performance in true neutrino energy',
             fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

## Step 5 -- Spectral comparison in reconstructed energy

The observable used in the oscillation fit is **reconstructed** neutrino energy
$E_\nu^\mathrm{reco}$, not the true energy.  The plots below show efficiency and
purity binned in $E_\nu^\mathrm{reco}$ for each working point.

Any improvement visible here directly translates to better sensitivity in the final
fit.  Compare with the NuMI reference (Triozzi, Shower TF Nov. 2025): nominal
efficiency ~20%, purity ~56% on NuMI; BNB results should be benchmarked against
those numbers accounting for the different flux shapes.

In [ ]:
RECO_COL = 'reco_nu_energy'
ok_reco  = df[RECO_COL].notna() & np.isfinite(df[RECO_COL]) & (df[RECO_COL] > 0)

E_BINS_R = np.array([0.0,0.2,0.4,0.6,0.8,1.0,1.2,1.4,1.6,1.8,2.0,2.4,2.8,3.2,4.0])
E_CEN_R  = 0.5*(E_BINS_R[:-1]+E_BINS_R[1:])
E_WID_R  = np.diff(E_BINS_R)

fig, (ax_e, ax_p) = plt.subplots(1, 2, figsize=(16, 5.5), dpi=300,
                                   gridspec_kw={'wspace': 0.32})

denom_true, _ = np.histogram(df.loc[sig, 'true_nu_energy'].dropna(), bins=E_BINS_R)

for (label, mask), col in zip(CUT_STEPS_COMPARE, CMP_COLS):
    mr = mask & ok_reco

    # efficiency (reco numerator, true denominator)
    num_r, _ = np.histogram(df.loc[mr & sig, RECO_COL].dropna(), bins=E_BINS_R)
    eff     = np.where(denom_true > 0, num_r/denom_true, np.nan)
    eff_err = np.where(denom_true > 0, np.sqrt(eff*(1-eff)/np.where(denom_true>0,denom_true,1)), np.nan)
    tot_eff = (mask & sig).sum() / sig.sum()

    # purity (both reco)
    ns, _ = np.histogram(df.loc[mr & sig, RECO_COL].dropna(), bins=E_BINS_R)
    nt, _ = np.histogram(df.loc[mr,        RECO_COL].dropna(), bins=E_BINS_R)
    pur     = np.where(nt > 0, ns/nt, np.nan)
    pur_err = np.where(nt > 0, np.sqrt(pur*(1-pur)/np.where(nt>0,nt,1)), np.nan)
    tot_pur = (mask & sig).sum() / mask.sum() if mask.sum() > 0 else np.nan

    kw = dict(fmt='o', ms=4.5, lw=1.8, capsize=0, color=col,
              markeredgecolor='white', markeredgewidth=0.5, zorder=5)

    ax_e.fill_between(E_CEN_R, eff-eff_err, eff+eff_err, color=col, alpha=0.14, step='mid', zorder=2)
    ax_e.errorbar(E_CEN_R, eff, xerr=E_WID_R/2, label=f'{label} ({100*tot_eff:.1f}%)', **kw)

    ax_p.fill_between(E_CEN_R, pur-pur_err, pur+pur_err, color=col, alpha=0.14, step='mid', zorder=2)
    ax_p.errorbar(E_CEN_R, pur, xerr=E_WID_R/2, label=f'{label} ({100*tot_pur:.1f}%)', **kw)

for ax, col2_lab, ylabel in [
    (ax_e, 'Signal events',             'Selection efficiency'),
    (ax_p, 'Selected events (nominal)', 'Selection purity'),
]:
    ax2 = ax.twinx()
    ax2.hist(df.loc[sig & ok_reco, RECO_COL].dropna(), bins=E_BINS_R,
             color='#bdbdbd', alpha=0.20, zorder=0)
    ax2.set_ylabel(col2_lab, color='#999999', fontsize=11)
    ax2.tick_params(axis='y', colors='#999999'); ax2.set_ylim(bottom=0); ax2.grid(False)
    ax.axhline(1.0, ls='--', lw=1.4, color='#555555', zorder=5)
    ax.set_xlabel(r'$E_{\nu}^{\mathrm{reco}}$ [GeV]')
    ax.set_ylabel(ylabel)
    ax.set_xlim(0, 4); ax.set_ylim(0, 1.35)
    ax.legend(ncol=1, loc='upper right', fontsize=9)

ax_e.set_title(r'Efficiency vs $E_{\nu}^{\mathrm{reco}}$')
ax_p.set_title(r'Purity vs $E_{\nu}^{\mathrm{reco}}$')
fig.suptitle('Nominal vs retuned: binned performance in reconstructed neutrino energy',
             fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

---
## Step 6 -- New physics-motivated cuts (retuned v2)

The 1D and 2D scans pointed to four levers that have not yet been fully exploited:

| Cut | Motivation | Proposed change |
|---|---|---|
| **dE/dx lower bound** | Signal showers sit at ~2 MeV/cm; values below ~1 MeV/cm indicate a failed reconstruction, not a real shower | Add `dE/dx > 1.0 MeV/cm` floor |
| **Tighter conversion gap** | The 2D angle-gap map shows purity rises steeply below ~2.5 cm; the nominal 4 cm is too generous | Tighten to `gap < 2.5 cm` |
| **Muon veto via track length** | The $\chi^2_\mu$ frontier was flat -- the veto is nearly inert on BNB. A track-length ceiling directly targets long muon tracks | Replace or supplement with `longest_trk_len < L` |
| **Tighter track score** | Background has more weight at score 0.47-0.50; tightening to ~0.46 gives a small but clean purity gain | `score < 0.46` |

We scan each independently below to quantify the gain before combining them.

In [ ]:
# Scan each new cut independently, holding everything else at the retuned working point.
# Base selection for these scans: the retuned shower-energy selection (best purity so far).

SCORE_COL   = 'shower_track_score'
valid_score = df[SCORE_COL].notna() & (df[SCORE_COL] > 0)

sel_retuned_base = (
    sel_flash(df)
    & c_shower(df, RETUNED_SHOWER_ENERGY_MIN)
    & c_dedx(df)          # nominal upper bound only for now
    & c_angle(df)
    & c_gap(df)
    & c_proton(df)
    & c_0pi(df)
    & c_muon(df)
)

# ---- 1. dE/dx lower bound --------------------------------------------------
DEDX_LO_VALS = np.linspace(0.3, 2.5, 60)
def sel_dedx_lo(v):
    return sel_retuned_base & (df['shower_avail_dedx'].fillna(0) > v)

# ---- 2. Conversion gap upper bound -----------------------------------------
GAP_VALS = np.linspace(0.5, 5.0, 60)
def sel_gap_tight(v):
    return (
        sel_flash(df)
        & c_shower(df, RETUNED_SHOWER_ENERGY_MIN)
        & c_dedx(df)
        & c_angle(df)
        & c_gap(df, v)
        & c_proton(df)
        & c_0pi(df)
        & c_muon(df)
    )

# ---- 3. Longest track length ceiling (if branch exists) --------------------
TRK_LEN_COL = 'longest_trk_len'
has_trk_len = TRK_LEN_COL in df.columns and df[TRK_LEN_COL].notna().sum() > 50
TRK_VALS = np.linspace(20, 250, 60)
def sel_trk_len(v):
    if not has_trk_len:
        return sel_retuned_base
    return sel_retuned_base & (df[TRK_LEN_COL].fillna(9999) < v)

# ---- 4. Track score (already in retuned v1, scan tighter) ------------------
SCORE_VALS = np.linspace(
    np.nanpercentile(df.loc[valid_score, SCORE_COL], 1),
    0.50, 60
)
def sel_score_tight(v):
    return (
        valid_score
        & sel_flash(df)
        & c_shower(df, RETUNED_SHOWER_ENERGY_MIN)
        & (df[SCORE_COL] < v)
        & c_dedx(df)
        & c_angle(df)
        & c_gap(df)
        & c_proton(df)
        & c_0pi(df)
        & c_muon(df)
    )

# ---- Run scans -------------------------------------------------------------
new_scans = [
    {'label': 'dE/dx lower bound',    'xlabel': r'dE/dx min [MeV/cm]',       'vals': DEDX_LO_VALS,  'fn': sel_dedx_lo,   'base': 0.0,  'dir': 'hi'},
    {'label': 'Conv. gap tighter',    'xlabel': r'Conv. gap max [cm]',        'vals': GAP_VALS,      'fn': sel_gap_tight, 'base': SHOWER_CONVGAP_MAX, 'dir': 'lo'},
    {'label': 'Track score ceiling',  'xlabel': r'Track score max',           'vals': SCORE_VALS,    'fn': sel_score_tight,'base': 0.50, 'dir': 'lo'},
]
if has_trk_len:
    new_scans.insert(2, {'label': 'Longest track len.', 'xlabel': r'Longest track len. [cm]',
                          'vals': TRK_VALS, 'fn': sel_trk_len, 'base': None, 'dir': 'lo'})

new_scan_results = {}
new_best_pts     = {}

for s in new_scans:
    out = scan_1d(s['vals'], s['fn'], sig)
    new_scan_results[s['label']] = out
    allowed = out[out['Eff. (%)'] >= 0.85*base_eff]   # allow 15% eff loss for new cuts
    if len(allowed):
        row = allowed.sort_values('Pur. (%)', ascending=False).iloc[0]
        new_best_pts[s['label']] = {'v': row['Cut value'], 'eff': row['Eff. (%)'], 'pur': row['Pur. (%)']}

# ---- Figure: 2 rows (eff, purity) x N cols --------------------------------
n = len(new_scans)
NCOLS = [CRIMSON, GOLD, '#3f8f3f', '#c13fd4']

fig, axes = plt.subplots(2, n, figsize=(4.5*n, 8.5), dpi=300,
                          gridspec_kw={'hspace': 0.50, 'wspace': 0.38})
if n == 1:
    axes = axes.reshape(2, 1)

for col, (s, col_c) in enumerate(zip(new_scans, NCOLS)):
    out = new_scan_results[s['label']]
    bpt = new_best_pts.get(s['label'])

    for row, (ykey, ylabel) in enumerate([('Eff. (%)', 'Efficiency (%)'), ('Pur. (%)', 'Purity (%)')]):
        ax = axes[row, col]
        ax.plot(out['Cut value'], out[ykey], '-', color=col_c, lw=2.2, zorder=4)
        ax.fill_between(out['Cut value'], out[ykey], alpha=0.12, color=col_c, zorder=2)

        # baseline of the scan (retuned_base global eff/pur)
        rb_eff = (sel_retuned_base & sig).sum() / sig.sum() * 100
        rb_pur = (sel_retuned_base & sig).sum() / sel_retuned_base.sum() * 100
        ax.axhline(rb_eff if row==0 else rb_pur,
                   ls='--', lw=1.5, color='#888888', alpha=0.7,
                   label='Retuned base' if col==0 and row==0 else '_')

        if s['base'] is not None:
            ax.axvline(s['base'], ls='--', lw=1.6, color=CRIMSON, alpha=0.85,
                       label='Nominal val.' if col==0 and row==0 else '_')
        if bpt:
            ax.axvline(bpt['v'], ls=':', lw=1.6, color=GOLD, alpha=0.90,
                       label='85% eff. opt.' if col==0 and row==0 else '_')
            ax.scatter([bpt['v']], [bpt[ykey.replace('Eff. (%)','eff').replace('Pur. (%)','pur')]
                                    if 'eff' in ykey.lower() else bpt['pur']],
                       marker='D', s=55, color=GOLD, edgecolor='white', lw=0.7, zorder=7)

        ax.set_xlabel(s['xlabel'], fontsize=9)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_ylim(bottom=0)
        if row == 0:
            ax.set_title(s['label'], fontsize=10)
            if col == 0:
                ax.legend(fontsize=7.5, loc='best')

fig.suptitle('New cut scans (base = retuned shower-energy selection)', fontsize=13, y=1.01)
plt.show()

# Print best points
print('Best values (>=85% of nominal efficiency):')
for label, bpt in new_best_pts.items():
    print(f"  {label:30s}  cut={bpt['v']:.3f}   Eff={bpt['eff']:.1f}%   Pur={bpt['pur']:.1f}%")
if not has_trk_len:
    print(f'  (longest_trk_len not found in ntuple -- track-length scan skipped)')

### Delta diagnostics: change from nominal

The panels below show the **change** in efficiency and purity relative to the nominal
selection as each new cut is tightened.  A horizontal line at zero means no change;
positive $\Delta$Purity means a purity gain; negative $\Delta$Efficiency means a cost.
The teal band marks the range where the purity gain exceeds the efficiency loss
(i.e., purity improves more than efficiency degrades).

In [ ]:
# Baseline metrics from the full nominal selection
nom_eff = base_metrics['Eff. (%)']
nom_pur = base_metrics['Pur. (%)']

fig, axes = plt.subplots(2, len(new_scans), figsize=(4.5*len(new_scans), 8.0),
                          dpi=300, gridspec_kw={'hspace': 0.50, 'wspace': 0.38})
if len(new_scans) == 1:
    axes = axes.reshape(2, 1)

NCOLS = [CRIMSON, GOLD, '#3f8f3f', '#c13fd4']

for col, (s, col_c) in enumerate(zip(new_scans, NCOLS)):
    out = new_scan_results[s['label']]

    d_eff = out['Eff. (%)'] - nom_eff   # negative = costs efficiency
    d_pur = out['Pur. (%)'] - nom_pur   # positive = gains purity

    for row, (dy, ylabel, flip) in enumerate([
        (d_eff, r'$\Delta$ Efficiency (pp)',  True),   # flip: less negative = better
        (d_pur, r'$\Delta$ Purity (pp)',      False),
    ]):
        ax = axes[row, col]
        vals = out['Cut value'].values

        # shade region where purity gain > |efficiency loss|
        gain_gt_cost = (d_pur > -d_eff)
        ax.fill_between(vals,
                        np.minimum(dy, 0) if row == 0 else np.maximum(dy, 0),
                        0,
                        where=gain_gt_cost,
                        color='#008080', alpha=0.13, zorder=1,
                        label='Pur. gain > eff. cost' if row == 1 and col == 0 else '_')

        ax.plot(vals, dy, '-', color=col_c, lw=2.2, zorder=4)
        ax.fill_between(vals, dy, 0, color=col_c, alpha=0.10, zorder=2)
        ax.axhline(0, ls='-',  lw=1.2, color='#555555', zorder=3)

        # nominal cut value marker
        if s['base'] is not None:
            ax.axvline(s['base'], ls='--', lw=1.6, color=CRIMSON, alpha=0.85,
                       label='Nominal val.' if col == 0 and row == 0 else '_')

        # best-purity marker
        bpt = new_best_pts.get(s['label'])
        if bpt:
            ax.axvline(bpt['v'], ls=':', lw=1.6, color=GOLD, alpha=0.90,
                       label='85% eff. opt.' if col == 0 and row == 0 else '_')

        ax.set_xlabel(s['xlabel'], fontsize=9)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_ylim(min(dy.min(), -1) * 1.25, max(dy.max(), 1) * 1.25)
        if row == 0:
            ax.set_title(s['label'], fontsize=10)
            if col == 0:
                ax.legend(fontsize=7.5, loc='best')

        # zero-crossing annotation
        cross = vals[np.argmin(np.abs(dy))]
        ax.annotate(f'min |$\\Delta$| at {cross:.2f}',
                    xy=(cross, dy[np.argmin(np.abs(dy))]),
                    xytext=(cross + (vals[-1]-vals[0])*0.12, dy.max()*0.5),
                    fontsize=6.5, color=col_c,
                    arrowprops=dict(arrowstyle='->', color=col_c, lw=0.8))

fig.suptitle(r'$\Delta$ efficiency and $\Delta$ purity vs nominal (pp = percentage points)',
             fontsize=13, y=1.01)
plt.show()

### 3-way joint impact scan

Each point below is a unique combination of (dE/dx lower bound, conv. gap max, track score max).
The axes show $\Delta$efficiency and $\Delta$purity relative to the nominal selection.
Points are coloured by the value of one cut variable at a time (three panels), so you can
read off which variable is responsible for moving a cluster of points in a given direction.

The **Pareto frontier** (maximum $\Delta$purity for each level of $\Delta$efficiency) is
drawn as a black line -- any optimal working point must lie on or near this curve.
The teal star marks the recommended selection.

In [ ]:
# sel_recommended defined here so this cell can run independently of Step 7.
V_DEDX_LO_LOCAL = 1.0
def c_dedx_floor_local(df, lo=V_DEDX_LO_LOCAL, hi=SHOWER_DEDX_MAX):
    dx = df['shower_avail_dedx']
    return (dx.fillna(99) < hi) & (dx.fillna(0) > lo)

sel_recommended = (
    sel_flash(df)
    & c_shower(df, RETUNED_SHOWER_ENERGY_MIN)
    & c_dedx_floor_local(df)
    & c_angle(df)
    & c_gap(df)
    & c_proton(df)
    & c_0pi(df)
    & c_muon(df)
)

# 3-way joint scan: all combinations of (dE/dx_lo, gap_max, score_max)
# Each combination is evaluated against the nominal selection.

DX_LO_GRID  = np.linspace(0.0, 2.0, 12)
GAP_GRID    = np.linspace(0.8, 5.0, 12)
SCORE_GRID  = np.linspace(
    np.nanpercentile(df.loc[valid_score, SCORE_COL], 1), 0.50, 12)

# Store results: one row per combination
rows_3w = []
for dx_lo in DX_LO_GRID:
    for gap_v in GAP_GRID:
        for sc_v in SCORE_GRID:
            sel_c = (
                valid_score
                & sel_flash(df)
                & c_shower(df, RETUNED_SHOWER_ENERGY_MIN)
                & (df['shower_avail_dedx'].fillna(0)   > dx_lo)
                & (df['shower_avail_dedx'].fillna(99)  < SHOWER_DEDX_MAX)
                & c_angle(df)
                & c_gap(df, gap_v)
                & c_proton(df)
                & c_0pi(df)
                & (df[SCORE_COL] < sc_v)
                & c_muon(df)
            )
            n_sig_c = (sel_c & sig).sum()
            n_sel_c = sel_c.sum()
            eff_c = n_sig_c / sig.sum() * 100 if sig.sum() > 0 else np.nan
            pur_c = n_sig_c / n_sel_c  * 100 if n_sel_c > 0  else np.nan
            rows_3w.append({'dx_lo': dx_lo, 'gap': gap_v, 'score': sc_v,
                            'd_eff': eff_c - nom_eff,
                            'd_pur': pur_c - nom_pur,
                            'eff': eff_c, 'pur': pur_c})

df3 = pd.DataFrame(rows_3w).dropna()

# Pareto frontier: max d_pur for each d_eff quantile
df3_s = df3.sort_values('d_eff')
pareto_x, pareto_y = [], []
running_max = -np.inf
for _, row in df3_s.iterrows():
    if row['d_pur'] > running_max:
        running_max = row['d_pur']
        pareto_x.append(row['d_eff'])
        pareto_y.append(row['d_pur'])

# Recommended point
rec_eff  = (sel_recommended & sig).sum() / sig.sum() * 100
rec_pur  = (sel_recommended & sig).sum() / sel_recommended.sum() * 100 if sel_recommended.sum() > 0 else np.nan

# ---- Figure: 3 panels, coloured by each cut variable -----------------------
cut_vars = [
    ('dx_lo', r'dE/dx lower bound [MeV/cm]', DX_LO_GRID),
    ('gap',   r'Conv. gap max [cm]',          GAP_GRID),
    ('score', r'Track score max',             SCORE_GRID),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.8), dpi=300,
                          gridspec_kw={'wspace': 0.38})

for ax, (var, cbar_label, grid) in zip(axes, cut_vars):
    sc = ax.scatter(df3['d_eff'], df3['d_pur'],
                    c=df3[var], cmap=cm.managua,
                    s=18, alpha=0.55, zorder=3,
                    edgecolors='none')
    plt.colorbar(sc, ax=ax, label=cbar_label, fraction=0.046, pad=0.04)

    # Pareto frontier
    ax.plot(pareto_x, pareto_y, '-', color='black', lw=1.8,
            zorder=6, label='Pareto frontier')

    # Reference lines
    ax.axhline(0, ls='--', lw=1.0, color='#888888', zorder=4)
    ax.axvline(0, ls='--', lw=1.0, color='#888888', zorder=4)

    # Recommended point
    ax.scatter([rec_eff - nom_eff], [rec_pur - nom_pur],
               marker='*', s=280, color='#008080',
               edgecolor='white', lw=0.8, zorder=9,
               label='Recommended')

    # Retuned-E point (no score/gap/dE/dx changes)
    ret_mask = (
        sel_flash(df)
        & c_shower(df, RETUNED_SHOWER_ENERGY_MIN)
        & c_dedx(df) & c_angle(df) & c_gap(df)
        & c_proton(df) & c_0pi(df) & c_muon(df)
    )
    re_eff = (ret_mask & sig).sum() / sig.sum() * 100
    re_pur = (ret_mask & sig).sum() / ret_mask.sum() * 100 if ret_mask.sum() > 0 else np.nan
    ax.scatter([re_eff - nom_eff], [re_pur - nom_pur],
               marker='D', s=90, color=CRIMSON,
               edgecolor='white', lw=0.7, zorder=9,
               label=r'Retuned $E_{sh}$ only')

    ax.set_xlabel(r'$\Delta$ Efficiency (pp)', fontsize=11)
    ax.set_ylabel(r'$\Delta$ Purity (pp)', fontsize=11)
    ax.set_title(f'Coloured by: {cbar_label}', fontsize=10)
    ax.legend(fontsize=8, loc='lower left')

fig.suptitle(
    r'3-way joint scan: $\Delta$Efficiency vs $\Delta$Purity from nominal'
    '\n(each point = one combination of dE/dx floor, gap, score)',
    fontsize=12, y=1.02)
plt.show()

# ---- Summary: which variable moves the Pareto frontier? --------------------
print('Pareto-frontier points (max purity for each efficiency level):')
for x, y in zip(pareto_x, pareto_y):
    row = df3[(df3['d_eff'].round(2)==round(x,2)) & (df3['d_pur'].round(2)==round(y,2))].iloc[0]
    print(f'  dEff={x:+.1f} pp  dPur={y:+.1f} pp  |  '
          f'dx_lo={row["dx_lo"]:.2f}  gap={row["gap"]:.2f}  score={row["score"]:.3f}')

In [ ]:
# Check which shower-multiplicity or second-shower variables exist in the ntuple.
# A second-shower veto targets pi^0 -> gamma gamma, where one photon fakes the electron.

shower_like_cols = [c for c in df.columns
                    if any(k in c.lower() for k in
                           ['n_shower', 'nshower', 'n_gamma', 'second_shower',
                            'n_pi0', 'pi0', 'n_other_shower'])]

print('Shower-multiplicity-like branches found:')
for c in shower_like_cols:
    nn  = df[c].notna().sum()
    med = df[c].median() if nn > 0 else float('nan')
    print(f'  {c:40s}  non-null={nn:6,}  median={med}')

if not shower_like_cols:
    print('  None found -- second-shower veto not directly available.')
    print('  Proxy: n_other_particles already requires 0 non-electron, non-proton PFPs,')
    print('  which implicitly suppresses pi^0 events where both photons are reconstructed.')
    print('  To add a dedicated veto, you would need to expose a shower-multiplicity branch')
    print('  in the ntuple macro (CC1e0piSelection_MakeAll.C).')

### Retuned v2 selection

We now combine the individually validated improvements into a single "v2" working point.
The values below use the scan optima printed above -- update them if you re-run the scans.

The v2 selection applies, on top of the retuned shower-energy floor:
- a **dE/dx window** (lower + upper bound),
- a **tighter conversion gap**,
- a **tighter track score**,
- and, if available, a **longest-track-length ceiling** to replace the inert $\chi^2_\mu$ veto.

In [ ]:
# ---- v2 working-point values (update from scan output above) ---------------
V2_SHOWER_ENERGY_MIN = RETUNED_SHOWER_ENERGY_MIN   # keep the retuned energy floor
V2_DEDX_MIN          = 1.0     # new lower bound on dE/dx
V2_DEDX_MAX          = SHOWER_DEDX_MAX             # keep nominal upper bound
V2_GAP_MAX           = 2.5     # tighter than nominal 4.0 cm
V2_SCORE_MAX         = 0.46    # tighter than nominal 0.50
V2_TRK_LEN_MAX       = 100.0   # longest track length ceiling (cm)

SCORE_COL   = 'shower_track_score'
valid_score = df[SCORE_COL].notna() & (df[SCORE_COL] > 0)

# dE/dx window: require both a lower AND upper bound
def c_dedx_window(df, lo=V2_DEDX_MIN, hi=V2_DEDX_MAX):
    dx = df['shower_avail_dedx']
    return (dx.fillna(99) < hi) & (dx.fillna(0) > lo)

# Longest-track-length veto (replaces chi2 muon veto if branch exists)
TRK_LEN_COL = 'longest_trk_len'
has_trk_len = TRK_LEN_COL in df.columns and df[TRK_LEN_COL].notna().sum() > 50

def c_trk_len_veto(df, v=V2_TRK_LEN_MAX):
    if not has_trk_len:
        return pd.Series(True, index=df.index)   # no-op if branch missing
    return df[TRK_LEN_COL].fillna(9999) < v

sel_v2 = (
    valid_score
    & sel_flash(df)
    & c_shower(df, V2_SHOWER_ENERGY_MIN)
    & c_dedx_window(df)
    & c_angle(df)
    & c_gap(df, V2_GAP_MAX)
    & c_proton(df)
    & c_0pi(df)
    & (df[SCORE_COL] < V2_SCORE_MAX)
    & c_trk_len_veto(df)   # replaces chi2 veto if track-length branch exists
)

# ---- Collect all configs for comparison ------------------------------------
CUT_STEPS_COMPARE = list(configs_retuned.items()) + [('Retuned v2', sel_v2)]

# ---- Global metrics table --------------------------------------------------
configs_v2 = {**configs_retuned, 'Retuned v2': sel_v2}
print('Global metrics -- all working points:')
tab = make_metrics_table(configs_v2, sig)
print(tab.to_string(index=False))
tab

In [ ]:
# Efficiency and purity vs E_nu^true -- all four working points.
# Four colours: blue (nominal), crimson (retuned-E), gold (retuned-E+score), purple (v2).

CMP_COLS_V2 = [BLUE, CRIMSON, GOLD, '#8b2fc9']

E_BINS_V = np.array([0.0,0.2,0.4,0.6,0.8,1.0,1.2,1.4,1.6,1.8,2.0,2.4,2.8,3.2,4.0])
E_CEN_V  = 0.5*(E_BINS_V[:-1]+E_BINS_V[1:])
E_WID_V  = np.diff(E_BINS_V)

fig, (ax_e, ax_p) = plt.subplots(1, 2, figsize=(16, 5.5), dpi=300,
                                   gridspec_kw={'wspace': 0.32})

denom, _ = np.histogram(df.loc[sig, 'true_nu_energy'].dropna(), bins=E_BINS_V)

for (label, mask), col in zip(CUT_STEPS_COMPARE, CMP_COLS_V2):
    num, _    = np.histogram(df.loc[mask & sig, 'true_nu_energy'].dropna(), bins=E_BINS_V)
    eff       = np.where(denom>0, num/denom, np.nan)
    eff_err   = np.where(denom>0, np.sqrt(eff*(1-eff)/np.where(denom>0,denom,1)), np.nan)
    tot_eff   = (mask & sig).sum() / sig.sum()

    ns, _     = np.histogram(df.loc[mask & sig, 'true_nu_energy'].dropna(), bins=E_BINS_V)
    nt, _     = np.histogram(df.loc[mask,        'true_nu_energy'].dropna(), bins=E_BINS_V)
    pur       = np.where(nt>0, ns/nt, np.nan)
    pur_err   = np.where(nt>0, np.sqrt(pur*(1-pur)/np.where(nt>0,nt,1)), np.nan)
    tot_pur   = (mask & sig).sum() / mask.sum() if mask.sum()>0 else np.nan

    kw = dict(fmt='o', ms=4.5, lw=1.8, capsize=0, color=col,
              markeredgecolor='white', markeredgewidth=0.5, zorder=5)
    lw_v  = 2.5 if 'v2' in label.lower() else 1.8

    ax_e.fill_between(E_CEN_V, eff-eff_err, eff+eff_err, color=col, alpha=0.13, step='mid', zorder=2)
    ax_e.errorbar(E_CEN_V, eff, xerr=E_WID_V/2,
                  label=f'{label} (eff {100*tot_eff:.1f}%)', **{**kw,'lw':lw_v})

    ax_p.fill_between(E_CEN_V, pur-pur_err, pur+pur_err, color=col, alpha=0.13, step='mid', zorder=2)
    ax_p.errorbar(E_CEN_V, pur, xerr=E_WID_V/2,
                  label=f'{label} (pur {100*tot_pur:.1f}%)', **{**kw,'lw':lw_v})

for ax, col2_lab, ylabel in [
    (ax_e, 'Signal events',             'Selection efficiency'),
    (ax_p, 'Selected events (nominal)', 'Selection purity'),
]:
    ax2 = ax.twinx()
    ax2.hist(df.loc[sig, 'true_nu_energy'].dropna(), bins=E_BINS_V,
             color='#bdbdbd', alpha=0.18, zorder=0)
    ax2.set_ylabel(col2_lab, color='#999999', fontsize=11)
    ax2.tick_params(axis='y', colors='#999999'); ax2.set_ylim(bottom=0); ax2.grid(False)
    ax.axhline(1.0, ls='--', lw=1.4, color='#555555', zorder=5)
    ax.set_xlabel(r'$E_{\nu}^{\mathrm{true}}$ [GeV]')
    ax.set_ylabel(ylabel)
    ax.set_xlim(0, 4); ax.set_ylim(0, 1.35)
    ax.legend(ncol=1, loc='upper right', fontsize=8.5)

ax_e.set_title(r'Efficiency vs $E_{\nu}^{\mathrm{true}}$')
ax_p.set_title(r'Purity vs $E_{\nu}^{\mathrm{true}}$')
fig.suptitle('All working points: binned performance in true neutrino energy',
             fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Same comparison in reconstructed energy -- the analysis observable.

RECO_COL = 'reco_nu_energy'
ok_reco  = df[RECO_COL].notna() & np.isfinite(df[RECO_COL]) & (df[RECO_COL] > 0)

fig, (ax_e, ax_p) = plt.subplots(1, 2, figsize=(16, 5.5), dpi=300,
                                   gridspec_kw={'wspace': 0.32})

denom_true, _ = np.histogram(df.loc[sig, 'true_nu_energy'].dropna(), bins=E_BINS_V)

for (label, mask), col in zip(CUT_STEPS_COMPARE, CMP_COLS_V2):
    mr = mask & ok_reco

    num_r, _ = np.histogram(df.loc[mr & sig, RECO_COL].dropna(), bins=E_BINS_V)
    eff      = np.where(denom_true>0, num_r/denom_true, np.nan)
    eff_err  = np.where(denom_true>0, np.sqrt(eff*(1-eff)/np.where(denom_true>0,denom_true,1)), np.nan)
    tot_eff  = (mask & sig).sum() / sig.sum()

    ns, _    = np.histogram(df.loc[mr & sig, RECO_COL].dropna(), bins=E_BINS_V)
    nt, _    = np.histogram(df.loc[mr,        RECO_COL].dropna(), bins=E_BINS_V)
    pur      = np.where(nt>0, ns/nt, np.nan)
    pur_err  = np.where(nt>0, np.sqrt(pur*(1-pur)/np.where(nt>0,nt,1)), np.nan)
    tot_pur  = (mask & sig).sum() / mask.sum() if mask.sum()>0 else np.nan

    lw_v = 2.5 if 'v2' in label.lower() else 1.8
    kw   = dict(fmt='o', ms=4.5, lw=lw_v, capsize=0, color=col,
                markeredgecolor='white', markeredgewidth=0.5, zorder=5)

    ax_e.fill_between(E_CEN_V, eff-eff_err, eff+eff_err, color=col, alpha=0.13, step='mid', zorder=2)
    ax_e.errorbar(E_CEN_V, eff, xerr=E_WID_V/2,
                  label=f'{label} (eff {100*tot_eff:.1f}%)', **kw)

    ax_p.fill_between(E_CEN_V, pur-pur_err, pur+pur_err, color=col, alpha=0.13, step='mid', zorder=2)
    ax_p.errorbar(E_CEN_V, pur, xerr=E_WID_V/2,
                  label=f'{label} (pur {100*tot_pur:.1f}%)', **kw)

for ax, col2_lab, ylabel in [
    (ax_e, 'Signal events',             'Selection efficiency'),
    (ax_p, 'Selected events (nominal)', 'Selection purity'),
]:
    ax2 = ax.twinx()
    ax2.hist(df.loc[sig & ok_reco, RECO_COL].dropna(), bins=E_BINS_V,
             color='#bdbdbd', alpha=0.18, zorder=0)
    ax2.set_ylabel(col2_lab, color='#999999', fontsize=11)
    ax2.tick_params(axis='y', colors='#999999'); ax2.set_ylim(bottom=0); ax2.grid(False)
    ax.axhline(1.0, ls='--', lw=1.4, color='#555555', zorder=5)
    ax.set_xlabel(r'$E_{\nu}^{\mathrm{reco}}$ [GeV]')
    ax.set_ylabel(ylabel)
    ax.set_xlim(0, 4); ax.set_ylim(0, 1.35)
    ax.legend(ncol=1, loc='upper right', fontsize=8.5)

ax_e.set_title(r'Efficiency vs $E_{\nu}^{\mathrm{reco}}$')
ax_p.set_title(r'Purity vs $E_{\nu}^{\mathrm{reco}}$')
fig.suptitle('All working points: binned performance in reconstructed neutrino energy',
             fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Punzi FOM and S/sqrt(S+B) summary -- quick visual to compare all working points.
# Punzi FOM = eff / (1 + sqrt(N_bkg)) targets 90% CL sensitivity in a counting experiment.

tab_v2 = make_metrics_table(configs_v2, sig).set_index('Selection')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), dpi=300,
                          gridspec_kw={'wspace': 0.40})

labels = list(configs_v2.keys())
x      = np.arange(len(labels))
BARCOLS = CMP_COLS_V2

for ax, metric, title in [
    (axes[0], 'Punzi FOM',       'Punzi FOM (higher = better sensitivity)'),
    (axes[1], r'$S/\sqrt{S+B}$', r'$S/\sqrt{S+B}$ (statistical significance proxy)'),
]:
    vals = [tab_v2.loc[l, metric] for l in labels]
    bars = ax.bar(x, vals, color=BARCOLS, alpha=0.85, edgecolor='white', linewidth=0.8, zorder=3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + max(vals)*0.01,
                f'{v:.4f}' if 'Punzi' in metric else f'{v:.2f}',
                ha='center', va='bottom', fontsize=8.5, fontweight='bold',
                color=bar.get_facecolor())
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=18, ha='right', fontsize=9)
    ax.set_ylabel(title.split('(')[0].strip())
    ax.set_title(title)
    ax.set_ylim(0, max(vals)*1.20)

fig.suptitle('Figure of merit comparison -- all working points (BNB Pandora-only)',
             fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

---
## Step 7 -- Recommended working point: retuned shower energy + dE/dx floor

The FOM comparison tells us the v2 over-cuts: the gap tightening and tighter score remove too
much efficiency without a compensating purity gain, and both the Punzi FOM and
$S/\sqrt{S+B}$ fall back below the retuned-energy-only selection.

The one genuinely free improvement is the **dE/dx lower bound at ~1.0 MeV/cm**.
From the scan, efficiency is flat up to ~1.5 MeV/cm while purity rises ~3 percentage
points -- these are failed reconstructions, not real showers, and removing them costs nothing.

The recommended working point ("Recommended") is therefore:

$$E_{\mathrm{sh}} > 0.314 \text{ GeV}, \quad 1.0 < \mathrm{dE/dx} < 3.5 \text{ MeV/cm}$$

Everything else stays at the nominal value.

In [ ]:
# ---- Recommended selection: retuned shower energy + dE/dx floor only --------

V_DEDX_LO = 1.0   # free efficiency-neutral floor from the scan

def c_dedx_floor(df, lo=V_DEDX_LO, hi=SHOWER_DEDX_MAX):
    dx = df['shower_avail_dedx']
    return (dx.fillna(99) < hi) & (dx.fillna(0) > lo)

sel_recommended = (
    sel_flash(df)
    & c_shower(df, RETUNED_SHOWER_ENERGY_MIN)
    & c_dedx_floor(df)
    & c_angle(df)
    & c_gap(df)
    & c_proton(df)
    & c_0pi(df)
    & c_muon(df)
)

SCORE_COL   = 'shower_track_score'
valid_score = df[SCORE_COL].notna() & (df[SCORE_COL] > 0)

configs_final = {
    'Nominal':
        sel_full(df),

    rf'Retuned $E_{{sh}}>$ {RETUNED_SHOWER_ENERGY_MIN:.3f} GeV':
        sel_flash(df) & c_shower(df, RETUNED_SHOWER_ENERGY_MIN)
        & c_dedx(df) & c_angle(df) & c_gap(df)
        & c_proton(df) & c_0pi(df) & c_muon(df),

    rf'Retuned $E_{{sh}}>$ {RETUNED_COMBINED_SHOWER_ENERGY_MIN:.3f} GeV, score $<$ {RETUNED_SCORE_MAX:.3f}':
        valid_score & sel_flash(df)
        & c_shower(df, RETUNED_COMBINED_SHOWER_ENERGY_MIN)
        & (df[SCORE_COL] < RETUNED_SCORE_MAX)
        & c_dedx(df) & c_angle(df) & c_gap(df)
        & c_proton(df) & c_0pi(df) & c_muon(df),

    rf'Recommended ($E_{{sh}}>$ {RETUNED_SHOWER_ENERGY_MIN:.3f} GeV, dE/dx $>$ {V_DEDX_LO} MeV/cm)':
        sel_recommended,

    rf'Recommended + score ($E_{{sh}}>$ {RETUNED_COMBINED_SHOWER_ENERGY_MIN:.3f} GeV, score $<$ {RETUNED_SCORE_MAX:.3f}, dE/dx $>$ {V_DEDX_LO} MeV/cm)':
        valid_score & sel_flash(df)
        & c_shower(df, RETUNED_COMBINED_SHOWER_ENERGY_MIN)
        & (df[SCORE_COL] < RETUNED_SCORE_MAX)
        & c_dedx_floor(df)
        & c_angle(df) & c_gap(df)
        & c_proton(df) & c_0pi(df) & c_muon(df),

    'Retuned v2 (over-cut)':
        sel_v2,
}

print('Final metrics comparison:')
tab_final = make_metrics_table(configs_final, sig)
print(tab_final.to_string(index=False))
tab_final


In [ ]:
# FOM bar chart + spectral comparison -- five configs.
# Draw nominal last so it is never hidden behind other bands.

FINAL_COLS  = [BLUE, CRIMSON, GOLD, '#008080', '#3a9e6e', '#8b2fc9']
#              nominal  retE   retE+score  recommended  rec+score   v2
FINAL_STEPS = list(configs_final.items())
SHORT_LABELS = ['Nominal', r'Retuned $E_{sh}$',
                r'Retuned $E_{sh}$+score',
                'Recommended', 'Recommended+score', 'Retuned v2']

E_BINS_F = np.array([0.0,0.2,0.4,0.6,0.8,1.0,1.2,1.4,1.6,1.8,2.0,2.4,2.8,3.2,4.0])
E_CEN_F  = 0.5*(E_BINS_F[:-1]+E_BINS_F[1:])
E_WID_F  = np.diff(E_BINS_F)

RECO_COL = 'reco_nu_energy'
ok_reco  = df[RECO_COL].notna() & np.isfinite(df[RECO_COL]) & (df[RECO_COL] > 0)

# ---- FOM bar chart ---------------------------------------------------------
tab_f  = make_metrics_table(configs_final, sig).set_index('Selection')
x_f    = np.arange(len(configs_final))

fig_fom, axes_fom = plt.subplots(1, 2, figsize=(14, 4.8), dpi=300,
                                  gridspec_kw={'wspace': 0.40})

for ax, metric, ylabel, fmt in [
    (axes_fom[0], 'Punzi FOM',       'Punzi FOM',      '.4f'),
    (axes_fom[1], r'$S/\sqrt{S+B}$', r'$S/\sqrt{S+B}$','.2f'),
]:
    vals = [tab_f.loc[l, metric] for l in configs_final]
    for xi, (v, col, lbl) in enumerate(zip(vals, FINAL_COLS, SHORT_LABELS)):
        is_rec = xi == 3
        ax.bar(xi, v, color=col,
               alpha=1.0 if is_rec else 0.80,
               edgecolor='#004040' if is_rec else 'white',
               linewidth=2.0 if is_rec else 0.8, zorder=3)
        ax.text(xi, v + max(vals)*0.012, f'{v:{fmt}}',
                ha='center', va='bottom', fontsize=8.5,
                fontweight='bold', color=col)
    ax.set_xticks(x_f)
    ax.set_xticklabels(SHORT_LABELS, rotation=18, ha='right', fontsize=8.5)
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, max(vals)*1.22)
    ax.set_title(f'{ylabel} by working point')

fig_fom.suptitle('Figure of merit -- recommended working point highlighted',
                  fontsize=12, y=1.02)
fig_fom.tight_layout()
plt.show()

# ---- Spectral comparison in E_reco (nominal drawn last so always visible) --
denom_true, _ = np.histogram(df.loc[sig, 'true_nu_energy'].dropna(), bins=E_BINS_F)

fig, (ax_e, ax_p) = plt.subplots(1, 2, figsize=(16, 5.5), dpi=300,
                                   gridspec_kw={'wspace': 0.32})

# Draw all non-nominal first, then nominal on top
draw_order = list(range(1, len(FINAL_STEPS))) + [0]

for idx in draw_order:
    label, mask = FINAL_STEPS[idx]
    col         = FINAL_COLS[idx]
    short       = SHORT_LABELS[idx]
    is_rec      = idx in (3, 4)
    is_nom      = idx == 0
    lw          = 2.8 if is_rec else (2.0 if is_nom else 1.5)
    alpha_b     = 0.22 if is_nom else (0.20 if is_rec else 0.10)
    zorder_line = 9 if is_nom else (8 if is_rec else 4)

    mr = mask & ok_reco
    num_r, _ = np.histogram(df.loc[mr & sig, RECO_COL].dropna(), bins=E_BINS_F)
    eff      = np.where(denom_true>0, num_r/denom_true, np.nan)
    eff_err  = np.where(denom_true>0, np.sqrt(eff*(1-eff)/np.where(denom_true>0,denom_true,1)), np.nan)
    tot_eff  = (mask & sig).sum() / sig.sum()

    ns, _ = np.histogram(df.loc[mr & sig, RECO_COL].dropna(), bins=E_BINS_F)
    nt, _ = np.histogram(df.loc[mr,        RECO_COL].dropna(), bins=E_BINS_F)
    pur      = np.where(nt>0, ns/nt, np.nan)
    pur_err  = np.where(nt>0, np.sqrt(pur*(1-pur)/np.where(nt>0,nt,1)), np.nan)
    tot_pur  = (mask & sig).sum() / mask.sum() if mask.sum()>0 else np.nan

    kw = dict(fmt='o', ms=5.0 if (is_rec or is_nom) else 3.8,
              lw=lw, capsize=0, color=col,
              markeredgecolor='white', markeredgewidth=0.6,
              zorder=zorder_line)

    ax_e.fill_between(E_CEN_F, eff-eff_err, eff+eff_err,
                      color=col, alpha=alpha_b, step='mid', zorder=zorder_line-1)
    ax_e.errorbar(E_CEN_F, eff, xerr=E_WID_F/2,
                  label=f'{short} (eff {100*tot_eff:.1f}%)', **kw)

    ax_p.fill_between(E_CEN_F, pur-pur_err, pur+pur_err,
                      color=col, alpha=alpha_b, step='mid', zorder=zorder_line-1)
    ax_p.errorbar(E_CEN_F, pur, xerr=E_WID_F/2,
                  label=f'{short} (pur {100*tot_pur:.1f}%)', **kw)

for ax, col2_lab, ylabel in [
    (ax_e, 'Signal events',             'Selection efficiency'),
    (ax_p, 'Selected events (nominal)', 'Selection purity'),
]:
    ax2 = ax.twinx()
    ax2.hist(df.loc[sig & ok_reco, RECO_COL].dropna(), bins=E_BINS_F,
             color='#bdbdbd', alpha=0.15, zorder=0)
    ax2.set_ylabel(col2_lab, color='#999999', fontsize=11)
    ax2.tick_params(axis='y', colors='#999999'); ax2.set_ylim(bottom=0); ax2.grid(False)
    ax.axhline(1.0, ls='--', lw=1.4, color='#555555', zorder=5)
    ax.set_xlabel(r'$E_{\nu}^{\mathrm{reco}}$ [GeV]')
    ax.set_ylabel(ylabel)
    ax.set_xlim(0, 4); ax.set_ylim(0, 1.35)
    ax.legend(ncol=1, loc='upper right', fontsize=8.5)

ax_e.set_title(r'Efficiency vs $E_{\nu}^{\mathrm{reco}}$')
ax_p.set_title(r'Purity vs $E_{\nu}^{\mathrm{reco}}$')
fig.suptitle(r'All working points: performance vs $E_{\nu}^{\mathrm{reco}}$',
             fontsize=13, y=1.02)
fig.tight_layout()
plt.show()


In [ ]:
# 2x2 diagnostic: Nominal vs Recommended+score (best working point).
# Rows: true energy / reco energy.   Cols: efficiency / purity.

E_BINS_2 = np.array([0.0,0.2,0.4,0.6,0.8,1.0,1.2,1.4,1.6,1.8,2.0,2.4,2.8,3.2,4.0])
E_CEN_2  = 0.5*(E_BINS_2[:-1]+E_BINS_2[1:])
E_WID_2  = np.diff(E_BINS_2)

RECO_COL = 'reco_nu_energy'
ok_reco  = df[RECO_COL].notna() & np.isfinite(df[RECO_COL]) & (df[RECO_COL] > 0)

# Recommended+score mask (mirrors configs_final definition)
sel_rec_score = (
    valid_score
    & sel_flash(df)
    & c_shower(df, RETUNED_COMBINED_SHOWER_ENERGY_MIN)
    & (df[SCORE_COL] < RETUNED_SCORE_MAX)
    & c_dedx_floor(df)
    & c_angle(df)
    & c_gap(df)
    & c_proton(df)
    & c_0pi(df)
    & c_muon(df)
)

PAIRS_2x2 = [
    ('Nominal',              sel_full(df),   BLUE),
    ('Recommended + score',  sel_rec_score,  '#3a9e6e'),
]

denom_true, _ = np.histogram(df.loc[sig, 'true_nu_energy'].dropna(), bins=E_BINS_2)

fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=300,
                          gridspec_kw={'hspace': 0.38, 'wspace': 0.32})

for row_idx, (energy_col, energy_label) in enumerate([
    ('true_nu_energy', r'$E_{\nu}^{\mathrm{true}}$ [GeV]'),
    (RECO_COL,         r'$E_{\nu}^{\mathrm{reco}}$ [GeV]'),
]):
    ok_e = df[energy_col].notna() & np.isfinite(df[energy_col]) & (df[energy_col] > 0)
    ok_  = ok_e & (ok_reco if row_idx == 1 else pd.Series(True, index=df.index))

    for col_idx, metric in enumerate(['efficiency', 'purity']):
        ax = axes[row_idx, col_idx]

        for sel_label, sel_mask, col_c in PAIRS_2x2:
            is_best = sel_label == 'Recommended + score'
            lw      = 2.5 if is_best else 1.8
            mr      = sel_mask & ok_

            if metric == 'efficiency':
                num, _ = np.histogram(df.loc[mr & sig, energy_col].dropna(), bins=E_BINS_2)
                denom  = denom_true
                y      = np.where(denom>0, num/denom, np.nan)
                yerr   = np.where(denom>0, np.sqrt(y*(1-y)/np.where(denom>0,denom,1)), np.nan)
                tot    = (sel_mask & sig).sum() / sig.sum()
                ylabel = 'Selection efficiency'
            else:
                ns, _ = np.histogram(df.loc[mr & sig, energy_col].dropna(), bins=E_BINS_2)
                nt, _ = np.histogram(df.loc[mr,        energy_col].dropna(), bins=E_BINS_2)
                y      = np.where(nt>0, ns/nt, np.nan)
                yerr   = np.where(nt>0, np.sqrt(y*(1-y)/np.where(nt>0,nt,1)), np.nan)
                tot    = (sel_mask & sig).sum() / sel_mask.sum() if sel_mask.sum()>0 else np.nan
                ylabel = 'Selection purity'

            pct_lbl = (f'eff {100*tot:.1f}%' if metric == 'efficiency'
                       else f'pur {100*tot:.1f}%')
            ax.fill_between(E_CEN_2, y-yerr, y+yerr,
                            color=col_c, alpha=0.20 if is_best else 0.12,
                            step='mid', zorder=2)
            ax.errorbar(E_CEN_2, y, xerr=E_WID_2/2,
                        fmt='o', ms=5.0 if is_best else 4.0, lw=lw, capsize=0,
                        color=col_c, markeredgecolor='white', markeredgewidth=0.6,
                        zorder=6 if is_best else 5,
                        label=f'{sel_label} ({pct_lbl})')

        ax2 = ax.twinx()
        ax2.hist(df.loc[sig & ok_, energy_col].dropna(),
                 bins=E_BINS_2, color='#bdbdbd', alpha=0.18, zorder=0)
        ax2.set_ylabel('Signal events', color='#999999', fontsize=10)
        ax2.tick_params(axis='y', colors='#999999'); ax2.set_ylim(bottom=0); ax2.grid(False)

        ax.axhline(1.0, ls='--', lw=1.3, color='#555555', zorder=5)
        ax.set_xlabel(energy_label)
        ax.set_ylabel(ylabel)
        ax.set_xlim(0, 4); ax.set_ylim(0, 1.35)
        ax.legend(fontsize=9.5, loc='upper right')

axes[0,0].set_title(r'Efficiency vs $E_{\nu}^{\mathrm{true}}$')
axes[0,1].set_title(r'Purity vs $E_{\nu}^{\mathrm{true}}$')
axes[1,0].set_title(r'Efficiency vs $E_{\nu}^{\mathrm{reco}}$')
axes[1,1].set_title(r'Purity vs $E_{\nu}^{\mathrm{reco}}$')

fig.suptitle('Nominal vs Recommended+score: full 2x2 diagnostic',
             fontsize=13, y=1.01)
fig.tight_layout()
plt.show()


In [ ]:
# Nominal vs Recommended+score: efficiency & purity vs E_reco,
# with a residual panel (Recommended+score - Nominal) beneath each column.

def _two_panel(energy_col, energy_label, ok_energy_extra=None):
    E_BINS = np.array([0.0,0.2,0.4,0.6,0.8,1.0,1.2,1.4,1.6,1.8,
                        2.0,2.4,2.8,3.2,4.0])
    E_CEN  = 0.5*(E_BINS[:-1]+E_BINS[1:])
    E_WID  = np.diff(E_BINS)

    ok_e = df[energy_col].notna() & np.isfinite(df[energy_col]) & (df[energy_col] > 0)
    ok_  = ok_e & (ok_energy_extra if ok_energy_extra is not None
                   else pd.Series(True, index=df.index))

    denom_true_loc, _ = np.histogram(df.loc[sig, 'true_nu_energy'].dropna(), bins=E_BINS)

    fig, axes = plt.subplots(
        2, 2, figsize=(16, 9), dpi=300,
        gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.08, 'wspace': 0.32}
    )

    PAIRS = [
        ('Nominal',             sel_full(df),   BLUE,      1.8, 4.0, 0.12),
        ('Recommended + score', sel_rec_score,  '#3a9e6e', 2.5, 5.0, 0.20),
    ]

    stored = {}

    for col_idx, metric in enumerate(['efficiency', 'purity']):
        ax_top = axes[0, col_idx]
        ax_res = axes[1, col_idx]
        stored[metric] = {}

        for sel_label, sel_mask, col_c, lw, ms, alpha_b in PAIRS:
            mr = sel_mask & ok_

            if metric == 'efficiency':
                num, _ = np.histogram(df.loc[mr & sig, energy_col].dropna(), bins=E_BINS)
                denom  = denom_true_loc
                y      = np.where(denom>0, num/denom, np.nan)
                yerr   = np.where(denom>0,
                                  np.sqrt(y*(1-y)/np.where(denom>0,denom,1)), np.nan)
                tot    = (sel_mask & sig).sum() / sig.sum()
                ylabel = 'Selection efficiency'
            else:
                ns, _ = np.histogram(df.loc[mr & sig, energy_col].dropna(), bins=E_BINS)
                nt, _ = np.histogram(df.loc[mr,        energy_col].dropna(), bins=E_BINS)
                y      = np.where(nt>0, ns/nt, np.nan)
                yerr   = np.where(nt>0,
                                  np.sqrt(y*(1-y)/np.where(nt>0,nt,1)), np.nan)
                tot    = (sel_mask & sig).sum() / sel_mask.sum() if sel_mask.sum()>0 else np.nan
                ylabel = 'Selection purity'

            stored[metric][sel_label] = (y, yerr)
            pct_lbl = (f'eff {100*tot:.1f}%' if metric == 'efficiency'
                       else f'pur {100*tot:.1f}%')

            ax_top.fill_between(E_CEN, y-yerr, y+yerr, color=col_c,
                                alpha=alpha_b, step='mid', zorder=2)
            ax_top.errorbar(E_CEN, y, xerr=E_WID/2,
                            fmt='o', ms=ms, lw=lw, capsize=0, color=col_c,
                            markeredgecolor='white', markeredgewidth=0.6,
                            zorder=6 if 'Rec' in sel_label else 5,
                            label=f'{sel_label} ({pct_lbl})')

        # backdrop histogram
        ax2 = ax_top.twinx()
        ax2.hist(df.loc[sig & ok_, energy_col].dropna(),
                 bins=E_BINS, color='#bdbdbd', alpha=0.15, zorder=0)
        ax2.set_ylabel('Signal events', color='#999999', fontsize=10)
        ax2.tick_params(axis='y', colors='#999999')
        ax2.set_ylim(bottom=0); ax2.grid(False)

        ax_top.axhline(1.0, ls='--', lw=1.3, color='#555555', zorder=5)
        ax_top.set_ylabel(ylabel)
        ax_top.set_xlim(0, 4); ax_top.set_ylim(0, 1.35)
        ax_top.legend(fontsize=9.5, loc='upper right')
        ax_top.set_xticklabels([])

        # ---- residual panel ------------------------------------------
        y_nom,  yerr_nom  = stored[metric]['Nominal']
        y_rec,  yerr_rec  = stored[metric]['Recommended + score']

        resid     = y_rec - y_nom
        resid_err = np.sqrt(np.where(np.isnan(yerr_rec), 0, yerr_rec)**2 +
                            np.where(np.isnan(yerr_nom), 0, yerr_nom)**2)

        ax_res.fill_between(
            E_CEN,
            np.where(np.isnan(resid), np.nan, resid - resid_err),
            np.where(np.isnan(resid), np.nan, resid + resid_err),
            color='#3a9e6e', alpha=0.18, step='mid', zorder=2)
        ax_res.errorbar(E_CEN, resid, xerr=E_WID/2,
                        fmt='o', ms=4.0, lw=1.8, capsize=0,
                        color='#3a9e6e', markeredgecolor='white',
                        markeredgewidth=0.5, zorder=5,
                        label='(Rec+score) - Nominal')
        ax_res.axhline(0, ls='--', lw=1.2, color='#555555', zorder=5)
        ax_res.set_xlabel(energy_label)
        res_ylabel = (r'$\Delta$ efficiency' if metric == 'efficiency'
                      else r'$\Delta$ purity')
        ax_res.set_ylabel(res_ylabel, fontsize=9)
        ax_res.set_xlim(0, 4)
        finite_resid = resid[np.isfinite(resid)]
        yabs = np.nanmax(np.abs(finite_resid)) if len(finite_resid) > 0 else 0.1
        ax_res.set_ylim(-yabs*1.6, yabs*1.6)
        ax_res.legend(fontsize=8.5, loc='upper right')

    axes[0,0].set_title(f'Efficiency vs {energy_label}')
    axes[0,1].set_title(f'Purity vs {energy_label}')
    return fig


# ---- E_reco version ----------------------------------------------------
RECO_COL = 'reco_nu_energy'
ok_reco  = df[RECO_COL].notna() & np.isfinite(df[RECO_COL]) & (df[RECO_COL] > 0)

fig_reco = _two_panel(
    energy_col='reco_nu_energy',
    energy_label=r'$E_{\nu}^{\mathrm{reco}}$ [GeV]',
    ok_energy_extra=ok_reco,
)
fig_reco.suptitle(
    r'Nominal vs Recommended+score: performance vs $E_{\nu}^{\mathrm{reco}}$'
    '\n(bottom: improvement = Recommended+score $-$ Nominal)',
    fontsize=12, y=1.02
)
fig_reco.tight_layout()
plt.show()


In [ ]:
# Same comparison but vs true neutrino energy.

fig_true = _two_panel(
    energy_col='true_nu_energy',
    energy_label=r'$E_{\nu}^{\mathrm{true}}$ [GeV]',
    ok_energy_extra=None,
)
fig_true.suptitle(
    r'Nominal vs Recommended+score: performance vs $E_{\nu}^{\mathrm{true}}$'
    '\n(bottom: improvement = Recommended+score $-$ Nominal)',
    fontsize=12, y=1.02
)
fig_true.tight_layout()
plt.show()


In [ ]:
# 2x2 cut-flow waterfall for the Recommended+score working point.
# Mirrors the style of the cumulative efficiency/purity plots (CUT_STEPS)
# but with the retuned cut values applied at each stage.
#
# Cut chain:
#   Presel. -> FM -> Electron ID (retuned E) -> dE/dx (floor+ceil) ->
#   Angle, gap -> Proton ID -> 0pi -> LE-mu veto -> score cut
#
# Rows: Etrue / Ereco.   Cols: efficiency / purity.
#
# Efficiency denominator:
#   Etrue row: all true signal events, binned in Etrue  (standard)
#   Ereco row: true signal events with valid reco, binned in Ereco
#              (so numerator and denominator are in the same observable)

E_BINS_CF = np.array([0.0,0.2,0.4,0.6,0.8,1.0,1.2,1.4,1.6,1.8,
                       2.0,2.4,2.8,3.2,4.0])
E_CEN_CF  = 0.5*(E_BINS_CF[:-1]+E_BINS_CF[1:])
E_WID_CF  = np.diff(E_BINS_CF)

RECO_COL = 'reco_nu_energy'
ok_reco  = df[RECO_COL].notna() & np.isfinite(df[RECO_COL]) & (df[RECO_COL] > 0)

# Retuned cumulative selectors -- score is its own final step
def sel_presel_r(df):    return sel_presel(df)
def sel_flash_r(df):     return sel_presel_r(df)   & c_flash(df)
def sel_shower_r(df):    return sel_flash_r(df)    & c_shower(df, RETUNED_COMBINED_SHOWER_ENERGY_MIN)
def sel_dedx_r(df):      return sel_shower_r(df)   & c_dedx_floor(df)
def sel_anglegap_r(df):  return sel_dedx_r(df)     & c_angle(df) & c_gap(df)
def sel_proton_r(df):    return sel_anglegap_r(df) & c_proton(df)
def sel_0pi_r(df):       return sel_proton_r(df)   & c_0pi(df)
def sel_muon_r(df):      return sel_0pi_r(df)      & c_muon(df)
def sel_score_r(df):     return sel_muon_r(df)     & valid_score & (df[SCORE_COL] < RETUNED_SCORE_MAX)

# 9 steps: extend SEL_COLS with one extra colour for the score step
SEL_COLS_R = list(SEL_COLS) + ['#e05c00']

CUT_STEPS_R = [
    ('Presel.',                                                          sel_presel_r),
    ('FM',                                                               sel_flash_r),
    (rf'Electron ID ($E_{{sh}}>$ {RETUNED_COMBINED_SHOWER_ENERGY_MIN:.3f} GeV)', sel_shower_r),
    (rf'dE/dx ({V_DEDX_LO}$-${SHOWER_DEDX_MAX} MeV/cm)',               sel_dedx_r),
    ('Angle, gap',                                                       sel_anglegap_r),
    ('Proton ID',                                                        sel_proton_r),
    (r'$0\pi$',                                                          sel_0pi_r),
    (r'LE-$\mu$ veto',                                                   sel_muon_r),
    (rf'Score $<$ {RETUNED_SCORE_MAX:.3f}',                              sel_score_r),
]

# Denominators: Etrue uses all signal in true energy bins;
#               Ereco uses signal events that have a valid reco energy, binned in Ereco.
denom_true_cf, _ = np.histogram(df.loc[sig, 'true_nu_energy'].dropna(), bins=E_BINS_CF)
denom_reco_cf, _ = np.histogram(df.loc[sig & ok_reco, RECO_COL].dropna(), bins=E_BINS_CF)

fig, axes = plt.subplots(2, 2, figsize=(18, 11), dpi=300,
                          gridspec_kw={'hspace': 0.38, 'wspace': 0.34})

for row_idx, (energy_col, energy_label) in enumerate([
    ('true_nu_energy', r'$E_{\nu}^{\mathrm{true}}$ [GeV]'),
    (RECO_COL,         r'$E_{\nu}^{\mathrm{reco}}$ [GeV]'),
]):
    ok_e   = df[energy_col].notna() & np.isfinite(df[energy_col]) & (df[energy_col] > 0)
    ok_    = ok_e & (ok_reco if row_idx == 1 else pd.Series(True, index=df.index))
    denom  = denom_reco_cf if row_idx == 1 else denom_true_cf

    for col_idx, metric in enumerate(['efficiency', 'purity']):
        ax = axes[row_idx, col_idx]

        for (label, cut_fn), col_c in zip(CUT_STEPS_R, SEL_COLS_R):
            mask = cut_fn(df)
            mr   = mask & ok_

            if metric == 'efficiency':
                num, _ = np.histogram(df.loc[mr & sig, energy_col].dropna(), bins=E_BINS_CF)
                y      = np.where(denom>0, num/denom, np.nan)
                yerr   = np.where(denom>0,
                                  np.sqrt(y*(1-y)/np.where(denom>0,denom,1)), np.nan)
                tot    = (mask & sig).sum() / sig.sum()
                ylabel = 'Selection efficiency'
            else:
                ns, _ = np.histogram(df.loc[mr & sig, energy_col].dropna(), bins=E_BINS_CF)
                nt, _ = np.histogram(df.loc[mr,        energy_col].dropna(), bins=E_BINS_CF)
                y      = np.where(nt>0, ns/nt, np.nan)
                yerr   = np.where(nt>0,
                                  np.sqrt(y*(1-y)/np.where(nt>0,nt,1)), np.nan)
                tot    = (mask & sig).sum() / mask.sum() if mask.sum()>0 else np.nan
                ylabel = 'Selection purity'

            pct_lbl = (f'eff {100*tot:.0f}%' if metric == 'efficiency'
                       else f'pur {100*tot:.0f}%')
            ax.fill_between(E_CEN_CF, y-yerr, y+yerr,
                            color=col_c, alpha=0.14, step='mid', zorder=2)
            ax.errorbar(E_CEN_CF, y, xerr=E_WID_CF/2,
                        fmt='o', ms=4.5, lw=1.8, capsize=0, color=col_c,
                        markeredgecolor='white', markeredgewidth=0.5,
                        zorder=5, label=f'{label} ({pct_lbl})')

        # grey backdrop histogram
        ax2 = ax.twinx()
        bkg_data = (df.loc[sig & ok_, energy_col].dropna()
                    if metric == 'efficiency'
                    else df.loc[sel_score_r(df) & ok_, energy_col].dropna())
        bkg_lab  = 'Signal events' if metric == 'efficiency' else 'Selected events (rec+score)'
        ax2.hist(bkg_data, bins=E_BINS_CF, color='#bdbdbd', alpha=0.18, zorder=0)
        ax2.set_ylabel(bkg_lab, color='#999999', fontsize=9)
        ax2.tick_params(axis='y', colors='#999999')
        ax2.set_ylim(bottom=0); ax2.grid(False)

        ax.axhline(1.0, ls='--', lw=1.3, color='#555555', zorder=5)
        ax.set_xlabel(energy_label)
        ax.set_ylabel(ylabel)
        ax.set_xlim(0, 4); ax.set_ylim(0, 1.35)
        ax.legend(ncol=1, loc='upper right', fontsize=7.8)

axes[0,0].set_title(r'Efficiency vs $E_{\nu}^{\mathrm{true}}$')
axes[0,1].set_title(r'Purity vs $E_{\nu}^{\mathrm{true}}$')
axes[1,0].set_title(r'Efficiency vs $E_{\nu}^{\mathrm{reco}}$')
axes[1,1].set_title(r'Purity vs $E_{\nu}^{\mathrm{reco}}$')

fig.suptitle(
    'CC1e0pi cut-flow waterfall: Recommended+score working point'
    r' ($E_{\nu}^{\mathrm{true}}$ and $E_{\nu}^{\mathrm{reco}}$)',
    fontsize=13, y=1.01
)
fig.tight_layout()
plt.show()
